In [ ]:
print("REVIVE AI is starting...")

REVIVE AI is starting...


In [ ]:
!pip install -q pandas numpy scikit-learn xgboost joblib openai

In [ ]:
import os

folders = [
    "revive_ai",
    "revive_ai/data",
    "revive_ai/models",
    "revive_ai/agent"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project structure created!")

Project structure created!


In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 10000

data = pd.DataFrame({
    "amount": np.random.randint(100, 20000, n),

    "payment_method": np.random.choice(
        ["upi", "card", "netbanking", "wallet"],
        n
    ),

    "failure_reason": np.random.choice(
        [
            "insufficient_balance",
            "bank_timeout",
            "card_declined",
            "technical_error",
            "authentication_failed"
        ],
        n
    ),

    "previous_successful_payments": np.random.randint(0, 20, n),

    "previous_failed_payments": np.random.randint(0, 6, n),

    "customer_age_days": np.random.randint(1, 1000, n),

    "attempt_number": np.random.randint(1, 4, n),

    "hour": np.random.randint(0, 24, n)
})

data.head()

,amount,payment_method,failure_reason,previous_successful_payments,previous_failed_payments,customer_age_days,attempt_number,hour
0,15895,wallet,insufficient_balance,13,2,331,2,3
1,960,wallet,card_declined,3,3,580,3,22
2,5490,wallet,card_declined,6,2,922,2,23
3,12064,netbanking,authentication_failed,1,4,899,2,13
4,11384,wallet,insufficient_balance,13,0,760,1,18


In [ ]:
def recovery_probability(row):
    score = 0

    # Customer history
    score += row["previous_successful_payments"] * 0.08
    score -= row["previous_failed_payments"] * 0.10

    # Customer relationship
    if row["customer_age_days"] > 180:
        score += 0.20

    # Failure reason
    if row["failure_reason"] == "bank_timeout":
        score += 0.30

    elif row["failure_reason"] == "insufficient_balance":
        score += 0.10

    elif row["failure_reason"] == "technical_error":
        score += 0.20

    elif row["failure_reason"] == "card_declined":
        score -= 0.15

    elif row["failure_reason"] == "authentication_failed":
        score -= 0.25

    # Attempts
    score -= (row["attempt_number"] - 1) * 0.15

    # Transaction size
    if row["amount"] > 10000:
        score -= 0.10

    probability = 1 / (1 + np.exp(-score))

    return probability


data["recovery_probability"] = data.apply(
    recovery_probability,
    axis=1
)

data["recovered"] = np.random.binomial(
    1,
    data["recovery_probability"]
)

data.head()

,amount,payment_method,failure_reason,previous_successful_payments,previous_failed_payments,customer_age_days,attempt_number,hour,recovery_probability,recovered
0,15895,wallet,insufficient_balance,13,2,331,2,3,0.708890,0
1,960,wallet,card_declined,3,3,580,3,22,0.423115,0
2,5490,wallet,card_declined,6,2,922,2,23,0.544879,0
3,12064,netbanking,authentication_failed,1,4,899,2,13,0.349781,0
4,11384,wallet,insufficient_balance,13,0,760,1,18,0.775564,0


In [ ]:
data["recovered"].value_counts()

,count
recovered,
1,6095
0,3905


In [ ]:
data.to_csv(
    "revive_ai/data/transactions.csv",
    index=False
)

print("Dataset saved!")

Dataset saved!


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

In [ ]:
X = data[
    [
        "amount",
        "payment_method",
        "failure_reason",
        "previous_successful_payments",
        "previous_failed_payments",
        "customer_age_days",
        "attempt_number",
        "hour"
    ]
]

y = data["recovered"]

In [ ]:
categorical_features = [
    "payment_method",
    "failure_reason"
]

numeric_features = [
    "amount",
    "previous_successful_payments",
    "previous_failed_payments",
    "customer_age_days",
    "attempt_number",
    "hour"
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)

In [ ]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 8000
Testing samples: 2000


In [ ]:
pipeline.fit(X_train, y_train)

print("REVIVE AI model trained successfully!")

REVIVE AI model trained successfully!


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

predictions = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Model Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, predictions))

Model Accuracy: 0.6

Classification Report:
              precision    recall  f1-score   support

           0       0.49      0.54      0.51       781
           1       0.68      0.64      0.66      1219

    accuracy                           0.60      2000
   macro avg       0.59      0.59      0.59      2000
weighted avg       0.61      0.60      0.60      2000



In [ ]:
probabilities = pipeline.predict_proba(X_test)

print(probabilities[:5])

[[0.46575676 0.53424324]
 [0.3216313  0.6783687 ]
 [0.4680849  0.5319151 ]
 [0.61452437 0.38547563]
 [0.46261241 0.53738759]]


In [ ]:
new_payment = pd.DataFrame([
    {
        "amount": 2499,
        "payment_method": "upi",
        "failure_reason": "bank_timeout",
        "previous_successful_payments": 8,
        "previous_failed_payments": 1,
        "customer_age_days": 340,
        "attempt_number": 1,
        "hour": 14
    }
])

recovery_probability = pipeline.predict_proba(
    new_payment
)[0][1]

print(
    f"Recovery Probability: {recovery_probability * 100:.2f}%"
)

Recovery Probability: 58.20%


In [ ]:
import joblib

joblib.dump(
    pipeline,
    "revive_ai/models/recovery_model.pkl"
)

print("Model saved successfully!")

Model saved successfully!


In [ ]:
print("Model Accuracy:", accuracy)
print(classification_report(y_test, predictions))
print(
    f"Recovery Probability: {recovery_probability * 100:.2f}%"
)

Model Accuracy: 0.6
              precision    recall  f1-score   support

           0       0.49      0.54      0.51       781
           1       0.68      0.64      0.66      1219

    accuracy                           0.60      2000
   macro avg       0.59      0.59      0.59      2000
weighted avg       0.61      0.60      0.60      2000

Recovery Probability: 58.20%


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

In [ ]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

In [ ]:
logistic_model.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['payment_method',
                                                   'failure_reason']),
                                                 ('numeric', 'passthrough',
                                                  ['amount',
                                                   'previous_successful_payments',
                                                   'previous_failed_payments',
                                                   'customer_age_days',
                                                   'attempt_number',
                                                   'hour'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
preprocessor_scaled = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features
        )
    ]
)

In [ ]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_scaled),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

In [ ]:
logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

logistic_predictions = logistic_model.predict(X_test)

logistic_probability = logistic_model.predict_proba(
    X_test
)[:, 1]

logistic_accuracy = accuracy_score(
    y_test,
    logistic_predictions
)

logistic_auc = roc_auc_score(
    y_test,
    logistic_probability
)

print("Logistic Regression Accuracy:", logistic_accuracy)
print("Logistic Regression ROC-AUC:", logistic_auc)

Logistic Regression Accuracy: 0.6395
Logistic Regression ROC-AUC: 0.6367543766589394


In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBClassifier(
                n_estimators=300,
                max_depth=5,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                eval_metric="logloss"
            )
        )
    ]
)

In [ ]:
xgb_model.fit(X_train, y_train)

print("XGBoost trained successfully!")

XGBoost trained successfully!


In [ ]:
xgb_predictions = xgb_model.predict(X_test)

xgb_probability = xgb_model.predict_proba(
    X_test
)[:, 1]

xgb_accuracy = accuracy_score(
    y_test,
    xgb_predictions
)

xgb_auc = roc_auc_score(
    y_test,
    xgb_probability
)

print("XGBoost Accuracy:", xgb_accuracy)
print("XGBoost ROC-AUC:", xgb_auc)

XGBoost Accuracy: 0.625
XGBoost ROC-AUC: 0.6257884393391447


In [ ]:
print("\n========== REVIVE AI MODEL COMPARISON ==========")

print(f"Random Forest Accuracy : {accuracy:.3f}")
print(f"Logistic Accuracy      : {logistic_accuracy:.3f}")
print(f"XGBoost Accuracy       : {xgb_accuracy:.3f}")

print()

print(f"Logistic ROC-AUC       : {logistic_auc:.3f}")
print(f"XGBoost ROC-AUC        : {xgb_auc:.3f}")


========== REVIVE AI MODEL COMPARISON ==========
Random Forest Accuracy : 0.600
Logistic Accuracy      : 0.639
XGBoost Accuracy       : 0.625

Logistic ROC-AUC       : 0.637
XGBoost ROC-AUC        : 0.626


In [ ]:
import joblib

joblib.dump(
    logistic_model,
    "revive_ai/models/recovery_model.pkl"
)

print("Best model saved!")

Best model saved!


In [ ]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("OpenAI API key loaded successfully!")

OpenAI API key loaded successfully!


In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("OpenAI API key loaded successfully!")

OpenAI API key loaded successfully!


In [ ]:
def revive_decision(payment, recovery_probability):

    probability = recovery_probability
    amount = payment["amount"]
    failure = payment["failure_reason"]
    previous_success = payment["previous_successful_payments"]
    attempts = payment["attempt_number"]

    # High probability + temporary failure
    if probability >= 0.75:

        if failure in ["bank_timeout", "technical_error"]:
            return {
                "action": "retry_later",
                "delay_minutes": 30,
                "discount": 0,
                "reason": "High recovery probability and likely temporary failure."
            }

        elif failure == "insufficient_balance":
            return {
                "action": "generate_payment_link",
                "delay_minutes": 0,
                "discount": 0,
                "reason": "Customer is likely recoverable but balance may be temporarily insufficient."
            }

    # Medium probability
    elif probability >= 0.45:

        if previous_success >= 5:
            return {
                "action": "send_message",
                "delay_minutes": 15,
                "discount": 0,
                "reason": "Returning customer with moderate recovery probability."
            }

        else:
            return {
                "action": "generate_payment_link",
                "delay_minutes": 0,
                "discount": 0,
                "reason": "Moderate recovery probability; provide an alternative payment route."
            }

    # Low probability
    else:

        if amount > 5000:
            return {
                "action": "offer_discount",
                "delay_minutes": 0,
                "discount": 0.02,
                "reason": "Low recovery probability but high-value transaction justifies a small incentive."
            }

        return {
            "action": "abandon_recovery",
            "delay_minutes": 0,
            "discount": 0,
            "reason": "Low recovery probability and low expected recovery value."
        }

In [ ]:
payment = {
    "payment_id": "pay_demo_001",
    "amount": 2499,
    "payment_method": "upi",
    "failure_reason": "bank_timeout",
    "previous_successful_payments": 8,
    "previous_failed_payments": 1,
    "customer_age_days": 340,
    "attempt_number": 1,
    "hour": 14
}

In [ ]:
payment_df = pd.DataFrame([payment])

recovery_probability = logistic_model.predict_proba(
    payment_df
)[0][1]

print(
    f"Recovery Probability: "
    f"{recovery_probability * 100:.2f}%"
)

Recovery Probability: 70.24%


In [ ]:
decision = revive_decision(
    payment,
    recovery_probability
)

print("REVIVE AI DECISION")
print("==================")

for key, value in decision.items():
    print(f"{key}: {value}")

REVIVE AI DECISION
action: send_message
delay_minutes: 15
discount: 0
reason: Returning customer with moderate recovery probability.


In [ ]:
agent_result = {
    "payment_id": payment["payment_id"],
    "amount": payment["amount"],
    "recovery_probability": round(
        recovery_probability * 100, 2
    ),
    "recommended_action": decision["action"],
    "delay_minutes": decision["delay_minutes"],
    "discount": decision["discount"],
    "reason": decision["reason"],
    "status": "ACTION_REQUIRED"
}

agent_result

{'payment_id': 'pay_demo_001',
 'amount': 2499,
 'recovery_probability': np.float64(70.24),
 'recommended_action': 'send_message',
 'delay_minutes': 15,
 'discount': 0,
 'reason': 'Returning customer with moderate recovery probability.',
 'status': 'ACTION_REQUIRED'}

In [ ]:
import uuid
from datetime import datetime


def retry_payment(payment_id):
    """
    Simulates retrying a failed payment.
    """

    print(f"🔄 Retrying payment: {payment_id}")

    # Simulate payment outcome
    success = np.random.random() < 0.75

    if success:
        print("✅ Payment recovered successfully!")

        return {
            "status": "success",
            "payment_id": payment_id,
            "message": "Payment successfully recovered."
        }

    else:
        print("❌ Payment retry failed.")

        return {
            "status": "failed",
            "payment_id": payment_id,
            "message": "Payment retry failed."
        }


def generate_payment_link(amount):
    """
    Simulates generating a new payment link.
    """

    link_id = "plink_" + uuid.uuid4().hex[:8]

    payment_link = f"https://pay.revive.ai/{link_id}"

    print(f"🔗 Payment link generated: {payment_link}")

    return {
        "status": "success",
        "amount": amount,
        "payment_link": payment_link
    }


def send_message(payment, message):
    """
    Simulates sending a customer recovery message.
    """

    print("📩 Sending recovery message...")
    print(f"Customer payment: ₹{payment['amount']}")
    print(f"Message: {message}")

    return {
        "status": "sent",
        "channel": "WhatsApp",
        "message": message
    }


def offer_discount(payment, discount_percentage):
    """
    Simulates offering a discount.
    """

    discount_amount = (
        payment["amount"] * discount_percentage
    )

    final_amount = (
        payment["amount"] - discount_amount
    )

    print("🎁 Discount generated")
    print(f"Original amount: ₹{payment['amount']}")
    print(f"Discount: ₹{discount_amount:.2f}")
    print(f"Final amount: ₹{final_amount:.2f}")

    return {
        "status": "success",
        "discount_percentage": discount_percentage,
        "discount_amount": round(discount_amount, 2),
        "final_amount": round(final_amount, 2)
    }


def abandon_recovery(payment_id):
    """
    Stops recovery attempts.
    """

    print(f"⛔ Recovery abandoned for {payment_id}")

    return {
        "status": "abandoned",
        "payment_id": payment_id
    }

In [ ]:
message_result = send_message(
    payment,
    "We noticed your payment didn't go through. "
    "You can complete your payment securely using your "
    "preferred payment method."
)

print(message_result)

📩 Sending recovery message...
Customer payment: ₹2499
Message: We noticed your payment didn't go through. You can complete your payment securely using your preferred payment method.
{'status': 'sent', 'channel': 'WhatsApp', 'message': "We noticed your payment didn't go through. You can complete your payment securely using your preferred payment method."}


In [ ]:
def execute_recovery_action(payment, decision):

    action = decision["action"]

    print("\n🤖 REVIVE EXECUTING ACTION")
    print("==========================")

    if action == "retry_now":

        return retry_payment(
            payment["payment_id"]
        )

    elif action == "retry_later":

        print(
            f"⏰ Retry scheduled in "
            f"{decision['delay_minutes']} minutes"
        )

        return {
            "status": "scheduled",
            "action": "retry_payment",
            "delay_minutes": decision["delay_minutes"]
        }

    elif action == "generate_payment_link":

        return generate_payment_link(
            payment["amount"]
        )

    elif action == "send_message":

        message = (
            "Your recent payment could not be completed. "
            "Please try again using your preferred payment method."
        )

        return send_message(
            payment,
            message
        )

    elif action == "offer_discount":

        return offer_discount(
            payment,
            decision["discount"]
        )

    elif action == "abandon_recovery":

        return abandon_recovery(
            payment["payment_id"]
        )

    else:

        return {
            "status": "error",
            "message": "Unknown recovery action."
        }

In [ ]:
execution_result = execute_recovery_action(
    payment,
    decision
)

print("\nEXECUTION RESULT")
print("================")
print(execution_result)


🤖 REVIVE EXECUTING ACTION
📩 Sending recovery message...
Customer payment: ₹2499
Message: Your recent payment could not be completed. Please try again using your preferred payment method.

EXECUTION RESULT
{'status': 'sent', 'channel': 'WhatsApp', 'message': 'Your recent payment could not be completed. Please try again using your preferred payment method.'}


In [ ]:
from datetime import datetime

In [ ]:
import pandas as pd
import numpy as np

from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

print("All basic libraries loaded successfully!")

All basic libraries loaded successfully!


In [ ]:
print("logistic_model" in globals())
print("revive_decision" in globals())
print("execute_recovery_action" in globals())
print("data" in globals())

False
False
False
False


In [ ]:
# ==========================================
# REVIVE AI — MASTER SETUP
# ==========================================

import pandas as pd
import numpy as np

from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

print("REVIVE AI environment ready!")

REVIVE AI environment ready!


In [ ]:
print("logistic_model" in globals())
print("revive_decision" in globals())
print("execute_recovery_action" in globals())
print("data" in globals())

False
False
False
False


In [ ]:
# ============================================================
# REVIVE AI — MASTER BUILD CELL
# ============================================================

import pandas as pd
import numpy as np
import uuid

from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score


# ============================================================
# 1. GENERATE SYNTHETIC TRANSACTION DATA
# ============================================================

np.random.seed(42)

n = 10000

data = pd.DataFrame({
    "amount": np.random.randint(100, 20000, n),

    "payment_method": np.random.choice(
        ["upi", "card", "netbanking", "wallet"],
        n
    ),

    "failure_reason": np.random.choice(
        [
            "insufficient_balance",
            "bank_timeout",
            "card_declined",
            "technical_error",
            "authentication_failed"
        ],
        n
    ),

    "previous_successful_payments": np.random.randint(0, 20, n),

    "previous_failed_payments": np.random.randint(0, 6, n),

    "customer_age_days": np.random.randint(1, 1000, n),

    "attempt_number": np.random.randint(1, 4, n),

    "hour": np.random.randint(0, 24, n)
})


# ============================================================
# 2. GENERATE RECOVERY OUTCOME
# ============================================================

def generate_recovery_probability(row):

    score = 0

    score += row["previous_successful_payments"] * 0.08
    score -= row["previous_failed_payments"] * 0.10

    if row["customer_age_days"] > 180:
        score += 0.20

    if row["failure_reason"] == "bank_timeout":
        score += 0.30

    elif row["failure_reason"] == "insufficient_balance":
        score += 0.10

    elif row["failure_reason"] == "technical_error":
        score += 0.20

    elif row["failure_reason"] == "card_declined":
        score -= 0.15

    elif row["failure_reason"] == "authentication_failed":
        score -= 0.25

    score -= (row["attempt_number"] - 1) * 0.15

    if row["amount"] > 10000:
        score -= 0.10

    probability = 1 / (1 + np.exp(-score))

    return probability


data["true_recovery_probability"] = data.apply(
    generate_recovery_probability,
    axis=1
)

data["recovered"] = np.random.binomial(
    1,
    data["true_recovery_probability"]
)


# ============================================================
# 3. PREPARE ML DATA
# ============================================================

features = [
    "amount",
    "payment_method",
    "failure_reason",
    "previous_successful_payments",
    "previous_failed_payments",
    "customer_age_days",
    "attempt_number",
    "hour"
]

X = data[features]

y = data["recovered"]


categorical_features = [
    "payment_method",
    "failure_reason"
]

numeric_features = [
    "amount",
    "previous_successful_payments",
    "previous_failed_payments",
    "customer_age_days",
    "attempt_number",
    "hour"
]


preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),

        (
            "numeric",
            StandardScaler(),
            numeric_features
        )
    ]
)


# ============================================================
# 4. TRAIN LOGISTIC REGRESSION
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)


logistic_model.fit(
    X_train,
    y_train
)


predictions = logistic_model.predict(X_test)

probabilities = logistic_model.predict_proba(
    X_test
)[:, 1]


accuracy = accuracy_score(
    y_test,
    predictions
)

auc = roc_auc_score(
    y_test,
    probabilities
)


# ============================================================
# 5. REVIVE DECISION ENGINE
# ============================================================

def revive_decision(payment, recovery_probability):

    probability = recovery_probability
    amount = payment["amount"]
    failure = payment["failure_reason"]
    previous_success = payment["previous_successful_payments"]

    if probability >= 0.75:

        if failure in [
            "bank_timeout",
            "technical_error"
        ]:

            return {
                "action": "retry_later",
                "delay_minutes": 30,
                "discount": 0,
                "reason":
                "High recovery probability with a temporary failure."
            }

        elif failure == "insufficient_balance":

            return {
                "action": "generate_payment_link",
                "delay_minutes": 0,
                "discount": 0,
                "reason":
                "High recovery potential but customer may need to retry later."
            }


    elif probability >= 0.45:

        if previous_success >= 5:

            return {
                "action": "send_message",
                "delay_minutes": 15,
                "discount": 0,
                "reason":
                "Returning customer with moderate recovery probability."
            }

        else:

            return {
                "action": "generate_payment_link",
                "delay_minutes": 0,
                "discount": 0,
                "reason":
                "Moderate recovery probability with limited customer history."
            }


    if amount > 5000:

        return {
            "action": "offer_discount",
            "delay_minutes": 0,
            "discount": 0.02,
            "reason":
            "Low recovery probability but transaction value is high."
        }


    return {
        "action": "abandon_recovery",
        "delay_minutes": 0,
        "discount": 0,
        "reason":
        "Low recovery probability and low expected recovery value."
    }


# ============================================================
# 6. AGENT TOOLS
# ============================================================

def retry_payment(payment_id):

    print(f"🔄 Retrying payment {payment_id}")

    return {
        "status": "success",
        "payment_id": payment_id
    }


def generate_payment_link(amount):

    link_id = "plink_" + uuid.uuid4().hex[:8]

    link = f"https://pay.revive.ai/{link_id}"

    print(f"🔗 Payment link generated: {link}")

    return {
        "status": "success",
        "payment_link": link,
        "amount": amount
    }


def send_message(payment, message):

    print("📩 Recovery message sent")
    print(message)

    return {
        "status": "sent",
        "channel": "WhatsApp",
        "message": message
    }


def offer_discount(payment, discount_percentage):

    discount_amount = (
        payment["amount"] * discount_percentage
    )

    final_amount = (
        payment["amount"] - discount_amount
    )

    print(
        f"🎁 Discount offered: ₹{discount_amount:.2f}"
    )

    return {
        "status": "success",
        "discount_amount": round(discount_amount, 2),
        "final_amount": round(final_amount, 2)
    }


def abandon_recovery(payment_id):

    print(
        f"⛔ Recovery abandoned for {payment_id}"
    )

    return {
        "status": "abandoned",
        "payment_id": payment_id
    }


# ============================================================
# 7. TOOL EXECUTOR
# ============================================================

def execute_recovery_action(
    payment,
    decision
):

    action = decision["action"]

    print("\n🤖 REVIVE EXECUTING")
    print("===================")

    if action == "retry_now":

        return retry_payment(
            payment["payment_id"]
        )


    elif action == "retry_later":

        print(
            f"⏰ Retry scheduled in "
            f"{decision['delay_minutes']} minutes"
        )

        return {
            "status": "scheduled",
            "delay_minutes":
            decision["delay_minutes"]
        }


    elif action == "generate_payment_link":

        return generate_payment_link(
            payment["amount"]
        )


    elif action == "send_message":

        message = (
            "Your recent payment could not be completed. "
            "Please try again using your preferred payment method."
        )

        return send_message(
            payment,
            message
        )


    elif action == "offer_discount":

        return offer_discount(
            payment,
            decision["discount"]
        )


    elif action == "abandon_recovery":

        return abandon_recovery(
            payment["payment_id"]
        )


# ============================================================
# 8. RECOVERY LOG
# ============================================================

recovery_log = []


# ============================================================
# READY
# ============================================================

print("====================================")
print("✅ REVIVE AI READY")
print("====================================")

print(
    f"Transactions: {len(data)}"
)

print(
    f"Model Accuracy: {accuracy:.3f}"
)

print(
    f"ROC-AUC: {auc:.3f}"
)

✅ REVIVE AI READY
Transactions: 10000
Model Accuracy: 0.639
ROC-AUC: 0.637


In [ ]:
payment = {
    "payment_id": "pay_demo_001",
    "amount": 2499,
    "payment_method": "upi",
    "failure_reason": "bank_timeout",
    "previous_successful_payments": 8,
    "previous_failed_payments": 1,
    "customer_age_days": 340,
    "attempt_number": 1,
    "hour": 14
}


payment_df = pd.DataFrame([
    payment
])


recovery_probability = (
    logistic_model
    .predict_proba(payment_df)[0][1]
)


decision = revive_decision(
    payment,
    recovery_probability
)


execution_result = execute_recovery_action(
    payment,
    decision
)


event = {
    "timestamp":
        datetime.now().isoformat(),

    "payment_id":
        payment["payment_id"],

    "amount":
        payment["amount"],

    "recovery_probability":
        round(
            float(recovery_probability) * 100,
            2
        ),

    "action":
        decision["action"],

    "reason":
        decision["reason"],

    "execution_status":
        execution_result["status"]
}


recovery_log.append(event)


print("\n==============================")
print("REVIVE AI RESULT")
print("==============================")

print(
    f"Payment: {payment['payment_id']}"
)

print(
    f"Amount: ₹{payment['amount']}"
)

print(
    f"Recovery Probability: "
    f"{recovery_probability * 100:.2f}%"
)

print(
    f"AI Action: {decision['action']}"
)

print(
    f"Reason: {decision['reason']}"
)

print(
    f"Execution: "
    f"{execution_result['status']}"
)

print("\nAUDIT LOG")
print(event)


🤖 REVIVE EXECUTING
📩 Recovery message sent
Your recent payment could not be completed. Please try again using your preferred payment method.

REVIVE AI RESULT
Payment: pay_demo_001
Amount: ₹2499
Recovery Probability: 70.24%
AI Action: send_message
Reason: Returning customer with moderate recovery probability.
Execution: sent

AUDIT LOG
{'timestamp': '2026-09-03T17:49:01.545562', 'payment_id': 'pay_demo_001', 'amount': 2499, 'recovery_probability': 70.24, 'action': 'send_message', 'reason': 'Returning customer with moderate recovery probability.', 'execution_status': 'sent'}


In [ ]:
def estimate_action_probability(
    base_probability,
    action,
    failure_reason,
    previous_successful_payments
):
    """
    Simulates how different recovery actions affect
    the probability of recovering a failed payment.

    These uplift values are prototype assumptions.
    In production, they would be learned from historical
    A/B test data.
    """

    probability = base_probability

    if action == "retry_now":

        if failure_reason in [
            "bank_timeout",
            "technical_error"
        ]:
            probability += 0.10
        else:
            probability += 0.03


    elif action == "retry_later":

        if failure_reason in [
            "bank_timeout",
            "technical_error"
        ]:
            probability += 0.15
        else:
            probability += 0.05


    elif action == "generate_payment_link":

        probability += 0.12


    elif action == "send_message":

        probability += 0.10

        if previous_successful_payments >= 5:
            probability += 0.05


    elif action == "offer_discount":

        probability += 0.25


    # Never allow probability above 99%
    return min(probability, 0.99)

In [ ]:
ACTION_COSTS = {
    "retry_now": 0,
    "retry_later": 0,
    "generate_payment_link": 0,
    "send_message": 2,
    "offer_discount": 100,
    "abandon_recovery": 0
}

In [ ]:
def calculate_expected_value(
    payment,
    base_probability,
    action
):

    amount = payment["amount"]

    probability = estimate_action_probability(
        base_probability,
        action,
        payment["failure_reason"],
        payment["previous_successful_payments"]
    )

    cost = ACTION_COSTS[action]

    if action == "offer_discount":

        discount = amount * 0.02

        customer_pays = amount - discount

        expected_revenue = (
            probability * customer_pays
        )

        total_cost = cost

        expected_net_revenue = (
            expected_revenue - total_cost
        )

    elif action == "abandon_recovery":

        probability = 0

        expected_revenue = 0

        total_cost = 0

        expected_net_revenue = 0

    else:

        expected_revenue = (
            probability * amount
        )

        total_cost = cost

        expected_net_revenue = (
            expected_revenue - total_cost
        )


    return {
        "action": action,
        "probability": probability,
        "expected_revenue": expected_revenue,
        "cost": total_cost,
        "expected_net_revenue": expected_net_revenue
    }

In [ ]:
def optimize_recovery_action(
    payment,
    base_probability
):

    actions = [
        "retry_now",
        "retry_later",
        "generate_payment_link",
        "send_message",
        "offer_discount",
        "abandon_recovery"
    ]

    results = []

    for action in actions:

        result = calculate_expected_value(
            payment,
            base_probability,
            action
        )

        results.append(result)


    best_action = max(
        results,
        key=lambda x: x["expected_net_revenue"]
    )


    return results, best_action

In [ ]:
results, best_action = optimize_recovery_action(
    payment,
    recovery_probability
)

In [ ]:
print("======================================")
print("REVIVE AI — REVENUE OPTIMIZER")
print("======================================")

for result in results:

    print(
        f"\nAction: {result['action']}"
    )

    print(
        f"Recovery Probability: "
        f"{result['probability'] * 100:.2f}%"
    )

    print(
        f"Expected Revenue: "
        f"₹{result['expected_revenue']:.2f}"
    )

    print(
        f"Cost: "
        f"₹{result['cost']:.2f}"
    )

    print(
        f"Expected Net Revenue: "
        f"₹{result['expected_net_revenue']:.2f}"
    )


print("\n======================================")
print("🏆 OPTIMAL ACTION")
print("======================================")

print(
    f"Action: {best_action['action']}"
)

print(
    f"Expected Recovery Probability: "
    f"{best_action['probability'] * 100:.2f}%"
)

print(
    f"Expected Net Revenue: "
    f"₹{best_action['expected_net_revenue']:.2f}"
)

REVIVE AI — REVENUE OPTIMIZER

Action: retry_now
Recovery Probability: 80.24%
Expected Revenue: ₹2005.10
Cost: ₹0.00
Expected Net Revenue: ₹2005.10

Action: retry_later
Recovery Probability: 85.24%
Expected Revenue: ₹2130.05
Cost: ₹0.00
Expected Net Revenue: ₹2130.05

Action: generate_payment_link
Recovery Probability: 82.24%
Expected Revenue: ₹2055.08
Cost: ₹0.00
Expected Net Revenue: ₹2055.08

Action: send_message
Recovery Probability: 85.24%
Expected Revenue: ₹2130.05
Cost: ₹2.00
Expected Net Revenue: ₹2128.05

Action: offer_discount
Recovery Probability: 95.24%
Expected Revenue: ₹2332.36
Cost: ₹100.00
Expected Net Revenue: ₹2232.36

Action: abandon_recovery
Recovery Probability: 0.00%
Expected Revenue: ₹0.00
Cost: ₹0.00
Expected Net Revenue: ₹0.00

🏆 OPTIMAL ACTION
Action: offer_discount
Expected Recovery Probability: 95.24%
Expected Net Revenue: ₹2232.36


In [ ]:
def explain_optimization(
    payment,
    best_action,
    results
):

    sorted_results = sorted(
        results,
        key=lambda x: x["expected_net_revenue"],
        reverse=True
    )

    winner = sorted_results[0]

    runner_up = sorted_results[1]

    explanation = f"""
REVIVE AI ANALYSIS

Transaction value:
₹{payment['amount']}

Failure reason:
{payment['failure_reason']}

Base recovery probability:
{recovery_probability * 100:.2f}%

Recommended action:
{winner['action']}

Expected recovery probability:
{winner['probability'] * 100:.2f}%

Expected net revenue:
₹{winner['expected_net_revenue']:.2f}

Next best alternative:
{runner_up['action']}

Why:
REVIVE selected {winner['action']} because it produces
the highest expected net revenue among the available
recovery strategies.

The agent avoids unnecessary discounts when another
recovery strategy is economically better.
"""

    return explanation

In [ ]:
print(
    explain_optimization(
        payment,
        best_action,
        results
    )
)


REVIVE AI ANALYSIS

Transaction value:
₹2499

Failure reason:
bank_timeout

Base recovery probability:
70.24%

Recommended action:
offer_discount

Expected recovery probability:
95.24%

Expected net revenue:
₹2232.36

Next best alternative:
retry_later

Why:
REVIVE selected offer_discount because it produces
the highest expected net revenue among the available
recovery strategies.

The agent avoids unnecessary discounts when another
recovery strategy is economically better.



In [ ]:
def revive_smart_decision(
    payment,
    recovery_probability
):

    results, best_action = optimize_recovery_action(
        payment,
        recovery_probability
    )

    return {
        "action": best_action["action"],
        "recovery_probability": best_action["probability"],
        "expected_revenue": best_action["expected_revenue"],
        "cost": best_action["cost"],
        "expected_net_revenue": best_action[
            "expected_net_revenue"
        ],
        "reason": (
            "Selected the action with the highest "
            "expected net revenue."
        ),
        "all_options": results
    }

In [ ]:
smart_decision = revive_smart_decision(
    payment,
    recovery_probability
)

print(smart_decision)

{'action': 'offer_discount', 'recovery_probability': np.float64(0.9523627906499029), 'expected_revenue': np.float64(2332.355521557425), 'cost': 100, 'expected_net_revenue': np.float64(2232.355521557425), 'reason': 'Selected the action with the highest expected net revenue.', 'all_options': [{'action': 'retry_now', 'probability': np.float64(0.8023627906499029), 'expected_revenue': np.float64(2005.1046138341073), 'cost': 0, 'expected_net_revenue': np.float64(2005.1046138341073)}, {'action': 'retry_later', 'probability': np.float64(0.852362790649903), 'expected_revenue': np.float64(2130.0546138341074), 'cost': 0, 'expected_net_revenue': np.float64(2130.0546138341074)}, {'action': 'generate_payment_link', 'probability': np.float64(0.8223627906499029), 'expected_revenue': np.float64(2055.0846138341076), 'cost': 0, 'expected_net_revenue': np.float64(2055.0846138341076)}, {'action': 'send_message', 'probability': np.float64(0.852362790649903), 'expected_revenue': np.float64(2130.0546138341074

In [ ]:
agent_context = {
    "payment": payment,

    "recovery_probability": float(
        recovery_probability
    ),

    "optimization": smart_decision
}

print("REVIVE agent context created.")

REVIVE agent context created.


In [ ]:
print(agent_context)

{'payment': {'payment_id': 'pay_demo_001', 'amount': 2499, 'payment_method': 'upi', 'failure_reason': 'bank_timeout', 'previous_successful_payments': 8, 'previous_failed_payments': 1, 'customer_age_days': 340, 'attempt_number': 1, 'hour': 14}, 'recovery_probability': 0.7023627906499029, 'optimization': {'action': 'offer_discount', 'recovery_probability': np.float64(0.9523627906499029), 'expected_revenue': np.float64(2332.355521557425), 'cost': 100, 'expected_net_revenue': np.float64(2232.355521557425), 'reason': 'Selected the action with the highest expected net revenue.', 'all_options': [{'action': 'retry_now', 'probability': np.float64(0.8023627906499029), 'expected_revenue': np.float64(2005.1046138341073), 'cost': 0, 'expected_net_revenue': np.float64(2005.1046138341073)}, {'action': 'retry_later', 'probability': np.float64(0.852362790649903), 'expected_revenue': np.float64(2130.0546138341074), 'cost': 0, 'expected_net_revenue': np.float64(2130.0546138341074)}, {'action': 'generate_

In [ ]:
def revive_agent(
    payment,
    recovery_probability
):

    # Evaluate all available strategies
    results, best_action = optimize_recovery_action(
        payment,
        recovery_probability
    )

    # Create the agent's decision
    decision = {
        "payment_id": payment["payment_id"],

        "action": best_action["action"],

        "recovery_probability": round(
            best_action["probability"],
            4
        ),

        "expected_revenue": round(
            best_action["expected_revenue"],
            2
        ),

        "expected_net_revenue": round(
            best_action["expected_net_revenue"],
            2
        ),

        "cost": round(
            best_action["cost"],
            2
        ),

        "reason": (
            f"The {best_action['action']} strategy "
            f"has the highest expected net revenue."
        )
    }

    return decision

In [ ]:
agent_decision = revive_agent(
    payment,
    recovery_probability
)

print("================================")
print("🤖 REVIVE AI AGENT")
print("================================")

for key, value in agent_decision.items():
    print(f"{key}: {value}")

🤖 REVIVE AI AGENT
payment_id: pay_demo_001
action: offer_discount
recovery_probability: 0.9524
expected_revenue: 2332.36
expected_net_revenue: 2232.36
cost: 100
reason: The offer_discount strategy has the highest expected net revenue.


In [ ]:
REVIVE_POLICY = {
    "max_discount": 0.05,
    "max_attempts": 3,
    "minimum_recovery_probability": 0.15,
    "avoid_unnecessary_discount": True,
    "maximize_expected_net_revenue": True
}

print(REVIVE_POLICY)

{'max_discount': 0.05, 'max_attempts': 3, 'minimum_recovery_probability': 0.15, 'avoid_unnecessary_discount': True, 'maximize_expected_net_revenue': True}


In [ ]:
def enforce_policy(
    payment,
    decision,
    policy
):

    # Prevent excessive discounts
    if decision["action"] == "offer_discount":

        if decision.get("discount", 0) > policy["max_discount"]:

            decision["action"] = "send_message"

            decision["reason"] = (
                "Discount exceeded merchant policy."
            )


    # Don't act when recovery probability is extremely low
    if (
        decision["recovery_probability"]
        < policy["minimum_recovery_probability"]
    ):

        decision["action"] = "abandon_recovery"

        decision["reason"] = (
            "Recovery probability is below "
            "the merchant's minimum threshold."
        )


    return decision

In [ ]:
safe_decision = enforce_policy(
    payment,
    agent_decision,
    REVIVE_POLICY
)

print(safe_decision)

{'payment_id': 'pay_demo_001', 'action': 'offer_discount', 'recovery_probability': np.float64(0.9524), 'expected_revenue': np.float64(2332.36), 'expected_net_revenue': np.float64(2232.36), 'cost': 100, 'reason': 'The offer_discount strategy has the highest expected net revenue.'}


In [ ]:
def build_agent_prompt(
    payment,
    recovery_probability,
    optimization_results
):

    prompt = f"""
You are REVIVE AI, an autonomous payment recovery agent.

Your objective is to recover failed payments while
maximizing the merchant's expected net revenue.

PAYMENT
-------
Payment ID: {payment['payment_id']}
Amount: ₹{payment['amount']}
Payment method: {payment['payment_method']}
Failure reason: {payment['failure_reason']}

CUSTOMER HISTORY
----------------
Previous successful payments:
{payment['previous_successful_payments']}

Previous failed payments:
{payment['previous_failed_payments']}

Customer age:
{payment['customer_age_days']} days

Attempt number:
{payment['attempt_number']}

ML RECOVERY PROBABILITY
-----------------------
{recovery_probability * 100:.2f}%

AVAILABLE STRATEGIES
--------------------
"""

    for result in optimization_results:

        prompt += f"""
Action: {result['action']}
Recovery probability: {result['probability'] * 100:.2f}%
Expected revenue: ₹{result['expected_revenue']:.2f}
Cost: ₹{result['cost']:.2f}
Expected net revenue: ₹{result['expected_net_revenue']:.2f}

"""

    prompt += """
DECISION RULES
--------------
1. Maximize expected net revenue.
2. Avoid unnecessary discounts.
3. Prefer non-discount recovery methods when economically equivalent.
4. Respect merchant policies.
5. Select exactly one action.

Return:
- selected action
- reasoning
- expected net revenue
- confidence
"""

    return prompt

In [ ]:
agent_prompt = build_agent_prompt(
    payment,
    recovery_probability,
    results
)

print(agent_prompt)


You are REVIVE AI, an autonomous payment recovery agent.

Your objective is to recover failed payments while
maximizing the merchant's expected net revenue.

PAYMENT
-------
Payment ID: pay_demo_001
Amount: ₹2499
Payment method: upi
Failure reason: bank_timeout

CUSTOMER HISTORY
----------------
Previous successful payments:
8

Previous failed payments:
1

Customer age:
340 days

Attempt number:
1

ML RECOVERY PROBABILITY
-----------------------
70.24%

AVAILABLE STRATEGIES
--------------------

Action: retry_now
Recovery probability: 80.24%
Expected revenue: ₹2005.10
Cost: ₹0.00
Expected net revenue: ₹2005.10


Action: retry_later
Recovery probability: 85.24%
Expected revenue: ₹2130.05
Cost: ₹0.00
Expected net revenue: ₹2130.05


Action: generate_payment_link
Recovery probability: 82.24%
Expected revenue: ₹2055.08
Cost: ₹0.00
Expected net revenue: ₹2055.08


Action: send_message
Recovery probability: 85.24%
Expected revenue: ₹2130.05
Cost: ₹2.00
Expected net revenue: ₹2128.05


Actio

In [ ]:
def simulated_llm_agent(
    optimization_results
):

    best = max(
        optimization_results,
        key=lambda x:
        x["expected_net_revenue"]
    )

    return {
        "action": best["action"],
        "expected_net_revenue":
            round(
                best["expected_net_revenue"],
                2
            ),
        "confidence": round(
            best["probability"],
            2
        ),
        "reason":
            f"Selected {best['action']} because "
            f"it has the highest expected net revenue."
    }

In [ ]:
llm_decision = simulated_llm_agent(
    results
)

print("🤖 SIMULATED LLM DECISION")
print("==========================")

for key, value in llm_decision.items():
    print(f"{key}: {value}")

🤖 SIMULATED LLM DECISION
action: offer_discount
expected_net_revenue: 2232.36
confidence: 0.95
reason: Selected offer_discount because it has the highest expected net revenue.


In [ ]:
final_decision = {
    "action": llm_decision["action"],
    "reason": llm_decision["reason"],
    "expected_net_revenue":
        llm_decision["expected_net_revenue"],
    "confidence":
        llm_decision["confidence"]
}

print(final_decision)

{'action': 'offer_discount', 'reason': 'Selected offer_discount because it has the highest expected net revenue.', 'expected_net_revenue': np.float64(2232.36), 'confidence': np.float64(0.95)}


In [ ]:
def execute_agent_decision(
    payment,
    agent_decision
):

    action = agent_decision["action"]

    print("\n==============================")
    print("🤖 REVIVE AGENT ACTION")
    print("==============================")

    print(
        f"Selected action: {action}"
    )

    print(
        f"Reason: {agent_decision['reason']}"
    )


    if action == "retry_now":

        return retry_payment(
            payment["payment_id"]
        )


    elif action == "retry_later":

        print(
            "⏰ Payment retry scheduled."
        )

        return {
            "status": "scheduled",
            "action": action
        }


    elif action == "generate_payment_link":

        return generate_payment_link(
            payment["amount"]
        )


    elif action == "send_message":

        return send_message(
            payment,
            "Your recent payment could not be completed. "
            "Please try again using your preferred payment method."
        )


    elif action == "offer_discount":

        return offer_discount(
            payment,
            0.02
        )


    elif action == "abandon_recovery":

        return abandon_recovery(
            payment["payment_id"]
        )


    return {
        "status": "error",
        "message": "Unknown action"
    }

In [ ]:
execution = execute_agent_decision(
    payment,
    final_decision
)

print("\nExecution result:")
print(execution)


🤖 REVIVE AGENT ACTION
Selected action: offer_discount
Reason: Selected offer_discount because it has the highest expected net revenue.
🎁 Discount offered: ₹49.98

Execution result:
{'status': 'success', 'discount_amount': 49.98, 'final_amount': 2449.02}


In [1]:
demo_payments = [
    {
        "payment_id": "pay_demo_101",
        "customer_name": "Rahul",
        "amount": 2499,
        "payment_method": "upi",
        "failure_reason": "bank_timeout",
        "previous_successful_payments": 8,
        "previous_failed_payments": 1,
        "customer_age_days": 340,
        "attempt_number": 1,
        "hour": 14
    },

    {
        "payment_id": "pay_demo_102",
        "customer_name": "Priya",
        "amount": 7999,
        "payment_method": "card",
        "failure_reason": "card_declined",
        "previous_successful_payments": 12,
        "previous_failed_payments": 2,
        "customer_age_days": 520,
        "attempt_number": 1,
        "hour": 18
    },

    {
        "payment_id": "pay_demo_103",
        "customer_name": "Arjun",
        "amount": 1299,
        "payment_method": "upi",
        "failure_reason": "insufficient_balance",
        "previous_successful_payments": 2,
        "previous_failed_payments": 1,
        "customer_age_days": 90,
        "attempt_number": 1,
        "hour": 11
    },

    {
        "payment_id": "pay_demo_104",
        "customer_name": "Sneha",
        "amount": 12499,
        "payment_method": "netbanking",
        "failure_reason": "technical_error",
        "previous_successful_payments": 15,
        "previous_failed_payments": 1,
        "customer_age_days": 720,
        "attempt_number": 1,
        "hour": 20
    },

    {
        "payment_id": "pay_demo_105",
        "customer_name": "Kabir",
        "amount": 499,
        "payment_method": "card",
        "failure_reason": "authentication_failed",
        "previous_successful_payments": 0,
        "previous_failed_payments": 4,
        "customer_age_days": 12,
        "attempt_number": 3,
        "hour": 2
    }
]

In [2]:
def process_failed_payment(payment):

    print("\n" + "=" * 60)
    print("🚨 PAYMENT FAILURE DETECTED")
    print("=" * 60)

    print(f"Payment ID: {payment['payment_id']}")
    print(f"Customer: {payment['customer_name']}")
    print(f"Amount: ₹{payment['amount']}")
    print(f"Method: {payment['payment_method']}")
    print(f"Failure: {payment['failure_reason']}")

    # -----------------------------------------
    # ML prediction
    # -----------------------------------------

    model_input = {
        key: payment[key]
        for key in [
            "amount",
            "payment_method",
            "failure_reason",
            "previous_successful_payments",
            "previous_failed_payments",
            "customer_age_days",
            "attempt_number",
            "hour"
        ]
    }

    payment_df = pd.DataFrame([model_input])

    recovery_probability = logistic_model.predict_proba(
        payment_df
    )[0][1]

    print("\n🧠 ML ANALYSIS")
    print(
        f"Base recovery probability: "
        f"{recovery_probability * 100:.2f}%"
    )

    # -----------------------------------------
    # Revenue optimization
    # -----------------------------------------

    results, best_action = optimize_recovery_action(
        payment,
        recovery_probability
    )

    print("\n💰 REVENUE OPTIMIZATION")

    for result in results:
        print(
            f"{result['action']:22} "
            f"| Probability: "
            f"{result['probability'] * 100:6.2f}% "
            f"| Net EV: ₹"
            f"{result['expected_net_revenue']:8.2f}"
        )

    # -----------------------------------------
    # Agent decision
    # -----------------------------------------

    agent_decision = simulated_llm_agent(
        results
    )

    print("\n🤖 REVIVE DECISION")
    print(f"Action: {agent_decision['action']}")
    print(f"Reason: {agent_decision['reason']}")
    print(
        f"Expected Net Revenue: "
        f"₹{agent_decision['expected_net_revenue']:.2f}"
    )

    # -----------------------------------------
    # Execute tool
    # -----------------------------------------

    execution = execute_agent_decision(
        payment,
        agent_decision
    )

    # -----------------------------------------
    # Record event
    # -----------------------------------------

    event = {
        "timestamp": datetime.now().isoformat(),
        "payment_id": payment["payment_id"],
        "customer_name": payment["customer_name"],
        "amount": payment["amount"],
        "failure_reason": payment["failure_reason"],
        "base_recovery_probability": round(
            float(recovery_probability) * 100,
            2
        ),
        "selected_action": agent_decision["action"],
        "expected_net_revenue": round(
            agent_decision["expected_net_revenue"],
            2
        ),
        "execution_status": execution["status"]
    }

    recovery_log.append(event)

    return event

In [4]:
import pandas as pd

In [5]:
import pandas as pd
import numpy as np
import uuid

from datetime import datetime

print("✅ Libraries loaded")

✅ Libraries loaded


In [6]:
print("logistic_model:", "logistic_model" in globals())
print("optimize_recovery_action:", "optimize_recovery_action" in globals())
print("simulated_llm_agent:", "simulated_llm_agent" in globals())
print("execute_agent_decision:", "execute_agent_decision" in globals())
print("demo_payments:", "demo_payments" in globals())

logistic_model: False
optimize_recovery_action: False
simulated_llm_agent: False
execute_agent_decision: False
demo_payments: True


In [8]:
# ============================================================
# REVIVE AI — REBUILD ML MODEL
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

np.random.seed(42)


# ------------------------------------------------------------
# 1. Generate transaction dataset
# ------------------------------------------------------------

n = 10000

data = pd.DataFrame({

    "amount": np.random.randint(100, 20000, n),

    "payment_method": np.random.choice(
        ["upi", "card", "netbanking", "wallet"],
        n
    ),

    "failure_reason": np.random.choice(
        [
            "insufficient_balance",
            "bank_timeout",
            "card_declined",
            "technical_error",
            "authentication_failed"
        ],
        n
    ),

    "previous_successful_payments":
        np.random.randint(0, 20, n),

    "previous_failed_payments":
        np.random.randint(0, 6, n),

    "customer_age_days":
        np.random.randint(1, 1000, n),

    "attempt_number":
        np.random.randint(1, 4, n),

    "hour":
        np.random.randint(0, 24, n)
})


# ------------------------------------------------------------
# 2. Generate recovery outcomes
# ------------------------------------------------------------

def generate_recovery_probability(row):

    score = 0

    score += (
        row["previous_successful_payments"] * 0.08
    )

    score -= (
        row["previous_failed_payments"] * 0.10
    )

    if row["customer_age_days"] > 180:
        score += 0.20

    if row["failure_reason"] == "bank_timeout":
        score += 0.30

    elif row["failure_reason"] == "insufficient_balance":
        score += 0.10

    elif row["failure_reason"] == "technical_error":
        score += 0.20

    elif row["failure_reason"] == "card_declined":
        score -= 0.15

    elif row["failure_reason"] == "authentication_failed":
        score -= 0.25

    score -= (
        row["attempt_number"] - 1
    ) * 0.15

    if row["amount"] > 10000:
        score -= 0.10

    return 1 / (1 + np.exp(-score))


data["true_recovery_probability"] = data.apply(
    generate_recovery_probability,
    axis=1
)

data["recovered"] = np.random.binomial(
    1,
    data["true_recovery_probability"]
)


# ------------------------------------------------------------
# 3. Prepare training data
# ------------------------------------------------------------

features = [
    "amount",
    "payment_method",
    "failure_reason",
    "previous_successful_payments",
    "previous_failed_payments",
    "customer_age_days",
    "attempt_number",
    "hour"
]

X = data[features]
y = data["recovered"]


categorical_features = [
    "payment_method",
    "failure_reason"
]

numeric_features = [
    "amount",
    "previous_successful_payments",
    "previous_failed_payments",
    "customer_age_days",
    "attempt_number",
    "hour"
]


preprocessor = ColumnTransformer(
    transformers=[

        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),

        (
            "numeric",
            StandardScaler(),
            numeric_features
        )
    ]
)


# ------------------------------------------------------------
# 4. Create and train model
# ------------------------------------------------------------

logistic_model = Pipeline(
    steps=[

        (
            "preprocessor",
            preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


logistic_model.fit(
    X_train,
    y_train
)


# ------------------------------------------------------------
# 5. Evaluate
# ------------------------------------------------------------

predictions = logistic_model.predict(
    X_test
)

probabilities = logistic_model.predict_proba(
    X_test
)[:, 1]


accuracy = accuracy_score(
    y_test,
    predictions
)

auc = roc_auc_score(
    y_test,
    probabilities
)


# ------------------------------------------------------------
# 6. Save model
# ------------------------------------------------------------

import joblib

joblib.dump(
    logistic_model,
    "revive_recovery_model.pkl"
)


# ------------------------------------------------------------
# DONE
# ------------------------------------------------------------

print("======================================")
print("✅ REVIVE ML MODEL REBUILT")
print("======================================")

print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")
print(f"Accuracy         : {accuracy:.3f}")
print(f"ROC-AUC          : {auc:.3f}")
print("Model saved      : revive_recovery_model.pkl")

✅ REVIVE ML MODEL REBUILT
Training samples : 8000
Testing samples  : 2000
Accuracy         : 0.639
ROC-AUC          : 0.637
Model saved      : revive_recovery_model.pkl


In [10]:
# ============================================================
# REVIVE AI — MASTER REVENUE OPTIMIZER
# ============================================================

# Cost assumptions for our prototype
ACTION_COSTS = {
    "retry_now": 0,
    "retry_later": 0,
    "generate_payment_link": 0,
    "send_message": 2,
    "offer_discount": 0,
    "abandon_recovery": 0
}


# ------------------------------------------------------------
# Estimate recovery probability after each action
# ------------------------------------------------------------

def estimate_action_probability(
    base_probability,
    action,
    failure_reason,
    previous_successful_payments
):

    probability = float(base_probability)

    if action == "retry_now":

        if failure_reason in [
            "bank_timeout",
            "technical_error"
        ]:
            probability += 0.10
        else:
            probability += 0.03


    elif action == "retry_later":

        if failure_reason in [
            "bank_timeout",
            "technical_error"
        ]:
            probability += 0.15
        else:
            probability += 0.05


    elif action == "generate_payment_link":

        probability += 0.12


    elif action == "send_message":

        probability += 0.10

        if previous_successful_payments >= 5:
            probability += 0.05


    elif action == "offer_discount":

        probability += 0.25


    elif action == "abandon_recovery":

        probability = 0


    # Keep probability between 0 and 0.99
    return min(max(probability, 0), 0.99)


# ------------------------------------------------------------
# Calculate expected revenue
# ------------------------------------------------------------

def calculate_expected_value(
    payment,
    base_probability,
    action
):

    amount = float(payment["amount"])

    probability = estimate_action_probability(
        base_probability,
        action,
        payment["failure_reason"],
        payment["previous_successful_payments"]
    )


    # Discount economics
    if action == "offer_discount":

        discount_percentage = 0.02

        customer_pays = (
            amount * (1 - discount_percentage)
        )

        expected_revenue = (
            probability * customer_pays
        )

        discount_cost = (
            amount * discount_percentage
        )

        expected_net_revenue = (
            expected_revenue
        )


        return {
            "action": action,
            "probability": probability,
            "expected_revenue": expected_revenue,
            "cost": discount_cost,
            "expected_net_revenue":
                expected_net_revenue
        }


    # Abandon recovery
    if action == "abandon_recovery":

        return {
            "action": action,
            "probability": 0,
            "expected_revenue": 0,
            "cost": 0,
            "expected_net_revenue": 0
        }


    # Normal action
    cost = ACTION_COSTS[action]

    expected_revenue = (
        probability * amount
    )

    expected_net_revenue = (
        expected_revenue - cost
    )


    return {
        "action": action,
        "probability": probability,
        "expected_revenue": expected_revenue,
        "cost": cost,
        "expected_net_revenue":
            expected_net_revenue
    }


# ------------------------------------------------------------
# Compare all possible recovery actions
# ------------------------------------------------------------

def optimize_recovery_action(
    payment,
    base_probability
):

    actions = [
        "retry_now",
        "retry_later",
        "generate_payment_link",
        "send_message",
        "offer_discount",
        "abandon_recovery"
    ]


    results = []


    for action in actions:

        result = calculate_expected_value(
            payment,
            base_probability,
            action
        )

        results.append(result)


    # Select highest expected net revenue
    best_action = max(
        results,
        key=lambda x:
        x["expected_net_revenue"]
    )


    return results, best_action


print("✅ REVIVE Revenue Optimizer loaded!")

✅ REVIVE Revenue Optimizer loaded!


In [11]:
def simulated_llm_agent(
    optimization_results
):

    best = max(
        optimization_results,
        key=lambda x:
        x["expected_net_revenue"]
    )

    return {
        "action": best["action"],

        "expected_net_revenue":
            float(
                best["expected_net_revenue"]
            ),

        "confidence":
            float(
                best["probability"]
            ),

        "reason":
            (
                f"Selected {best['action']} "
                f"because it has the highest "
                f"expected net revenue."
            )
    }


print("✅ REVIVE Agent loaded!")

✅ REVIVE Agent loaded!


In [12]:
print("retry_payment:", "retry_payment" in globals())
print(
    "generate_payment_link:",
    "generate_payment_link" in globals()
)
print(
    "send_message:",
    "send_message" in globals()
)
print(
    "offer_discount:",
    "offer_discount" in globals()
)
print(
    "abandon_recovery:",
    "abandon_recovery" in globals()
)

retry_payment: False
generate_payment_link: False
send_message: False
offer_discount: False
abandon_recovery: False


In [13]:
test_payment = demo_payments[0]

test_df = pd.DataFrame([{
    "amount": test_payment["amount"],
    "payment_method": test_payment["payment_method"],
    "failure_reason": test_payment["failure_reason"],
    "previous_successful_payments":
        test_payment["previous_successful_payments"],
    "previous_failed_payments":
        test_payment["previous_failed_payments"],
    "customer_age_days":
        test_payment["customer_age_days"],
    "attempt_number":
        test_payment["attempt_number"],
    "hour":
        test_payment["hour"]
}])

test_probability = logistic_model.predict_proba(
    test_df
)[0][1]

results, best_action = optimize_recovery_action(
    test_payment,
    test_probability
)

print("======================================")
print("REVIVE OPTIMIZER TEST")
print("======================================")

print(
    f"Base probability: "
    f"{test_probability * 100:.2f}%"
)

for result in results:

    print(
        f"{result['action']:25} "
        f"| Recovery: "
        f"{result['probability'] * 100:6.2f}% "
        f"| Expected Net: ₹"
        f"{result['expected_net_revenue']:8.2f}"
    )

print("\n🏆 BEST ACTION")
print(best_action["action"])

REVIVE OPTIMIZER TEST
Base probability: 70.24%
retry_now                 | Recovery:  80.24% | Expected Net: ₹ 2005.10
retry_later               | Recovery:  85.24% | Expected Net: ₹ 2130.05
generate_payment_link     | Recovery:  82.24% | Expected Net: ₹ 2055.08
send_message              | Recovery:  85.24% | Expected Net: ₹ 2128.05
offer_discount            | Recovery:  95.24% | Expected Net: ₹ 2332.36
abandon_recovery          | Recovery:   0.00% | Expected Net: ₹    0.00

🏆 BEST ACTION
offer_discount


In [15]:
# ============================================================
# REVIVE AI — AGENT ACTION EXECUTOR
# ============================================================

def execute_agent_decision(payment, agent_decision):

    action = agent_decision["action"]

    print("\n🤖 REVIVE AGENT EXECUTING")
    print("==========================")
    print(f"Selected action: {action}")

    # --------------------------------------------------------
    # 1. Retry immediately
    # --------------------------------------------------------

    if action == "retry_now":

        result = retry_payment(
            payment["payment_id"]
        )

        return result


    # --------------------------------------------------------
    # 2. Retry later
    # --------------------------------------------------------

    elif action == "retry_later":

        delay = 30

        print(
            f"⏰ Retry scheduled in {delay} minutes"
        )

        return {
            "status": "scheduled",
            "action": "retry_payment",
            "delay_minutes": delay
        }


    # --------------------------------------------------------
    # 3. Generate payment link
    # --------------------------------------------------------

    elif action == "generate_payment_link":

        result = generate_payment_link(
            payment["amount"]
        )

        return result


    # --------------------------------------------------------
    # 4. Send recovery message
    # --------------------------------------------------------

    elif action == "send_message":

        message = (
            f"Hi {payment.get('customer_name', 'there')}, "
            f"your payment of ₹{payment['amount']} "
            "could not be completed. "
            "Please try again using your preferred "
            "payment method."
        )

        result = send_message(
            payment,
            message
        )

        return result


    # --------------------------------------------------------
    # 5. Offer discount
    # --------------------------------------------------------

    elif action == "offer_discount":

        discount = 0.02

        result = offer_discount(
            payment,
            discount
        )

        return result


    # --------------------------------------------------------
    # 6. Abandon recovery
    # --------------------------------------------------------

    elif action == "abandon_recovery":

        result = abandon_recovery(
            payment["payment_id"]
        )

        return result


    # --------------------------------------------------------
    # Unknown action
    # --------------------------------------------------------

    else:

        return {
            "status": "error",
            "message": f"Unknown action: {action}"
        }


print("✅ Agent executor loaded!")

✅ Agent executor loaded!


In [17]:
# ============================================================
# REVIVE AI — MASTER ACTION TOOLS
# ============================================================

import uuid


def retry_payment(payment_id):

    print(f"🔄 Retrying payment: {payment_id}")

    return {
        "status": "success",
        "payment_id": payment_id,
        "message": "Payment retry initiated."
    }


def generate_payment_link(amount):

    link_id = "plink_" + uuid.uuid4().hex[:8]

    payment_link = (
        f"https://pay.revive.ai/{link_id}"
    )

    print("🔗 Payment link generated")
    print(f"Amount: ₹{amount}")
    print(f"Link: {payment_link}")

    return {
        "status": "success",
        "payment_link": payment_link,
        "amount": amount
    }


def send_message(payment, message):

    print("📩 Recovery message sent")
    print(f"Customer: {payment.get('customer_name', 'Customer')}")
    print(f"Message: {message}")

    return {
        "status": "sent",
        "channel": "WhatsApp",
        "message": message
    }


def offer_discount(payment, discount_percentage):

    original_amount = float(
        payment["amount"]
    )

    discount_amount = (
        original_amount *
        discount_percentage
    )

    final_amount = (
        original_amount -
        discount_amount
    )

    print("🎁 RECOVERY DISCOUNT OFFERED")
    print(
        f"Original amount: ₹{original_amount:.2f}"
    )
    print(
        f"Discount: ₹{discount_amount:.2f}"
    )
    print(
        f"Customer pays: ₹{final_amount:.2f}"
    )

    return {
        "status": "success",
        "original_amount": original_amount,
        "discount_percentage":
            discount_percentage,
        "discount_amount":
            round(discount_amount, 2),
        "final_amount":
            round(final_amount, 2)
    }


def abandon_recovery(payment_id):

    print(
        f"⛔ Recovery abandoned: {payment_id}"
    )

    return {
        "status": "abandoned",
        "payment_id": payment_id
    }


print("✅ All REVIVE action tools loaded!")

✅ All REVIVE action tools loaded!


In [18]:
result = process_failed_payment(
    demo_payments[0]
)


🚨 PAYMENT FAILURE DETECTED
Payment ID: pay_demo_101
Customer: Rahul
Amount: ₹2499
Method: upi
Failure: bank_timeout

🧠 ML ANALYSIS
Base recovery probability: 70.24%

💰 REVENUE OPTIMIZATION
retry_now              | Probability:  80.24% | Net EV: ₹ 2005.10
retry_later            | Probability:  85.24% | Net EV: ₹ 2130.05
generate_payment_link  | Probability:  82.24% | Net EV: ₹ 2055.08
send_message           | Probability:  85.24% | Net EV: ₹ 2128.05
offer_discount         | Probability:  95.24% | Net EV: ₹ 2332.36
abandon_recovery       | Probability:   0.00% | Net EV: ₹    0.00

🤖 REVIVE DECISION
Action: offer_discount
Reason: Selected offer_discount because it has the highest expected net revenue.
Expected Net Revenue: ₹2332.36

🤖 REVIVE AGENT EXECUTING
Selected action: offer_discount
🎁 RECOVERY DISCOUNT OFFERED
Original amount: ₹2499.00
Discount: ₹49.98
Customer pays: ₹2449.02


NameError: name 'recovery_log' is not defined

In [19]:
# ==========================================
# REVIVE AI — RECOVERY LOG
# ==========================================

recovery_log = []

print("✅ Recovery log initialized!")

✅ Recovery log initialized!


In [20]:
result = process_failed_payment(
    demo_payments[0]
)


🚨 PAYMENT FAILURE DETECTED
Payment ID: pay_demo_101
Customer: Rahul
Amount: ₹2499
Method: upi
Failure: bank_timeout

🧠 ML ANALYSIS
Base recovery probability: 70.24%

💰 REVENUE OPTIMIZATION
retry_now              | Probability:  80.24% | Net EV: ₹ 2005.10
retry_later            | Probability:  85.24% | Net EV: ₹ 2130.05
generate_payment_link  | Probability:  82.24% | Net EV: ₹ 2055.08
send_message           | Probability:  85.24% | Net EV: ₹ 2128.05
offer_discount         | Probability:  95.24% | Net EV: ₹ 2332.36
abandon_recovery       | Probability:   0.00% | Net EV: ₹    0.00

🤖 REVIVE DECISION
Action: offer_discount
Reason: Selected offer_discount because it has the highest expected net revenue.
Expected Net Revenue: ₹2332.36

🤖 REVIVE AGENT EXECUTING
Selected action: offer_discount
🎁 RECOVERY DISCOUNT OFFERED
Original amount: ₹2499.00
Discount: ₹49.98
Customer pays: ₹2449.02


In [21]:
recovery_log = []

for payment in demo_payments:
    process_failed_payment(payment)


🚨 PAYMENT FAILURE DETECTED
Payment ID: pay_demo_101
Customer: Rahul
Amount: ₹2499
Method: upi
Failure: bank_timeout

🧠 ML ANALYSIS
Base recovery probability: 70.24%

💰 REVENUE OPTIMIZATION
retry_now              | Probability:  80.24% | Net EV: ₹ 2005.10
retry_later            | Probability:  85.24% | Net EV: ₹ 2130.05
generate_payment_link  | Probability:  82.24% | Net EV: ₹ 2055.08
send_message           | Probability:  85.24% | Net EV: ₹ 2128.05
offer_discount         | Probability:  95.24% | Net EV: ₹ 2332.36
abandon_recovery       | Probability:   0.00% | Net EV: ₹    0.00

🤖 REVIVE DECISION
Action: offer_discount
Reason: Selected offer_discount because it has the highest expected net revenue.
Expected Net Revenue: ₹2332.36

🤖 REVIVE AGENT EXECUTING
Selected action: offer_discount
🎁 RECOVERY DISCOUNT OFFERED
Original amount: ₹2499.00
Discount: ₹49.98
Customer pays: ₹2449.02

🚨 PAYMENT FAILURE DETECTED
Payment ID: pay_demo_102
Customer: Priya
Amount: ₹7999
Method: card
Failure: ca

In [23]:
recovery_log = []

print("✅ Old recovery log cleared")

✅ Old recovery log cleared


In [24]:
def simulate_customer_outcome(
    payment,
    selected_action,
    optimization_results
):

    selected_result = next(
        result
        for result in optimization_results
        if result["action"] == selected_action
    )

    probability = selected_result["probability"]

    recovered = np.random.random() < probability

    if recovered:

        recovered_amount = float(payment["amount"])

        # If discount was offered,
        # customer pays 98% of original amount
        if selected_action == "offer_discount":
            recovered_amount *= 0.98

        print("\n🎉 CUSTOMER PAYMENT RECOVERED")
        print(
            f"Recovered revenue: "
            f"₹{recovered_amount:.2f}"
        )

        return {
            "recovered": True,
            "recovered_revenue":
                round(recovered_amount, 2)
        }

    else:

        print("\n❌ CUSTOMER DID NOT COMPLETE PAYMENT")

        return {
            "recovered": False,
            "recovered_revenue": 0
        }


print("✅ Customer outcome simulator loaded!")

✅ Customer outcome simulator loaded!


In [25]:
recovery_log = []

for payment in demo_payments:

    process_failed_payment(payment)


🚨 PAYMENT FAILURE DETECTED
Payment ID: pay_demo_101
Customer: Rahul
Amount: ₹2499
Method: upi
Failure: bank_timeout

🧠 ML ANALYSIS
Base recovery probability: 70.24%

💰 REVENUE OPTIMIZATION
retry_now              | Probability:  80.24% | Net EV: ₹ 2005.10
retry_later            | Probability:  85.24% | Net EV: ₹ 2130.05
generate_payment_link  | Probability:  82.24% | Net EV: ₹ 2055.08
send_message           | Probability:  85.24% | Net EV: ₹ 2128.05
offer_discount         | Probability:  95.24% | Net EV: ₹ 2332.36
abandon_recovery       | Probability:   0.00% | Net EV: ₹    0.00

🤖 REVIVE DECISION
Action: offer_discount
Reason: Selected offer_discount because it has the highest expected net revenue.
Expected Net Revenue: ₹2332.36

🤖 REVIVE AGENT EXECUTING
Selected action: offer_discount
🎁 RECOVERY DISCOUNT OFFERED
Original amount: ₹2499.00
Discount: ₹49.98
Customer pays: ₹2449.02

🚨 PAYMENT FAILURE DETECTED
Payment ID: pay_demo_102
Customer: Priya
Amount: ₹7999
Method: card
Failure: ca

In [26]:
recovery_df = pd.DataFrame(recovery_log)

print("Columns available:")
print(recovery_df.columns.tolist())

Columns available:
['timestamp', 'payment_id', 'customer_name', 'amount', 'failure_reason', 'base_recovery_probability', 'selected_action', 'expected_net_revenue', 'execution_status']


In [28]:
# ============================================================
# REVIVE AI — FINAL TRANSACTION PROCESSOR
# ============================================================

def process_failed_payment_final(payment):

    print("\n" + "=" * 60)
    print("🚨 PAYMENT FAILURE DETECTED")
    print("=" * 60)

    print(f"Payment ID: {payment['payment_id']}")
    print(f"Customer: {payment['customer_name']}")
    print(f"Amount: ₹{payment['amount']}")
    print(f"Method: {payment['payment_method']}")
    print(f"Failure: {payment['failure_reason']}")

    # --------------------------------------------------------
    # ML PREDICTION
    # --------------------------------------------------------

    model_input = {
        "amount": payment["amount"],
        "payment_method": payment["payment_method"],
        "failure_reason": payment["failure_reason"],
        "previous_successful_payments":
            payment["previous_successful_payments"],
        "previous_failed_payments":
            payment["previous_failed_payments"],
        "customer_age_days":
            payment["customer_age_days"],
        "attempt_number":
            payment["attempt_number"],
        "hour":
            payment["hour"]
    }

    payment_df = pd.DataFrame([model_input])

    base_probability = float(
        logistic_model.predict_proba(
            payment_df
        )[0][1]
    )

    print("\n🧠 ML ANALYSIS")
    print(
        f"Base recovery probability: "
        f"{base_probability * 100:.2f}%"
    )

    # --------------------------------------------------------
    # REVENUE OPTIMIZATION
    # --------------------------------------------------------

    results, best_action = optimize_recovery_action(
        payment,
        base_probability
    )

    print("\n💰 REVENUE OPTIMIZATION")

    for result in results:

        print(
            f"{result['action']:25} | "
            f"Probability: "
            f"{result['probability'] * 100:6.2f}% | "
            f"Net EV: ₹"
            f"{result['expected_net_revenue']:8.2f}"
        )

    # --------------------------------------------------------
    # AI AGENT
    # --------------------------------------------------------

    agent_decision = simulated_llm_agent(
        results
    )

    print("\n🤖 REVIVE DECISION")

    print(
        f"Action: {agent_decision['action']}"
    )

    print(
        f"Reason: {agent_decision['reason']}"
    )

    print(
        f"Expected Net Revenue: ₹"
        f"{agent_decision['expected_net_revenue']:.2f}"
    )

    # --------------------------------------------------------
    # EXECUTE ACTION
    # --------------------------------------------------------

    execution = execute_agent_decision(
        payment,
        agent_decision
    )

    # --------------------------------------------------------
    # SIMULATE CUSTOMER RESPONSE
    # --------------------------------------------------------

    customer_result = simulate_customer_outcome(
        payment,
        agent_decision["action"],
        results
    )

    # --------------------------------------------------------
    # CREATE CLEAN LOG RECORD
    # --------------------------------------------------------

    event = {

        "timestamp":
            datetime.now().isoformat(),

        "payment_id":
            payment["payment_id"],

        "customer_name":
            payment["customer_name"],

        "amount":
            float(payment["amount"]),

        "failure_reason":
            payment["failure_reason"],

        "base_recovery_probability":
            round(
                base_probability * 100,
                2
            ),

        "selected_action":
            agent_decision["action"],

        "expected_net_revenue":
            round(
                float(
                    agent_decision[
                        "expected_net_revenue"
                    ]
                ),
                2
            ),

        "execution_status":
            execution["status"],

        "recovered":
            bool(
                customer_result["recovered"]
            ),

        "recovered_revenue":
            float(
                customer_result[
                    "recovered_revenue"
                ]
            )
    }

    recovery_log.append(event)

    print("\n📋 TRANSACTION LOGGED")

    print(
        f"Recovered: "
        f"{'YES ✅' if event['recovered'] else 'NO ❌'}"
    )

    print(
        f"Recovered revenue: ₹"
        f"{event['recovered_revenue']:,.2f}"
    )

    return event


print("✅ Final transaction processor loaded!")

✅ Final transaction processor loaded!


In [29]:
recovery_log = []

print("✅ Old transaction log cleared")

✅ Old transaction log cleared


In [30]:
for payment in demo_payments:

    process_failed_payment_final(
        payment
    )


🚨 PAYMENT FAILURE DETECTED
Payment ID: pay_demo_101
Customer: Rahul
Amount: ₹2499
Method: upi
Failure: bank_timeout

🧠 ML ANALYSIS
Base recovery probability: 70.24%

💰 REVENUE OPTIMIZATION
retry_now                 | Probability:  80.24% | Net EV: ₹ 2005.10
retry_later               | Probability:  85.24% | Net EV: ₹ 2130.05
generate_payment_link     | Probability:  82.24% | Net EV: ₹ 2055.08
send_message              | Probability:  85.24% | Net EV: ₹ 2128.05
offer_discount            | Probability:  95.24% | Net EV: ₹ 2332.36
abandon_recovery          | Probability:   0.00% | Net EV: ₹    0.00

🤖 REVIVE DECISION
Action: offer_discount
Reason: Selected offer_discount because it has the highest expected net revenue.
Expected Net Revenue: ₹2332.36

🤖 REVIVE AGENT EXECUTING
Selected action: offer_discount
🎁 RECOVERY DISCOUNT OFFERED
Original amount: ₹2499.00
Discount: ₹49.98
Customer pays: ₹2449.02

🎉 CUSTOMER PAYMENT RECOVERED
Recovered revenue: ₹2449.02

📋 TRANSACTION LOGGED
Recovered

In [31]:
recovery_df = pd.DataFrame(
    recovery_log
)

print(
    recovery_df.columns.tolist()
)

['timestamp', 'payment_id', 'customer_name', 'amount', 'failure_reason', 'base_recovery_probability', 'selected_action', 'expected_net_revenue', 'execution_status', 'recovered', 'recovered_revenue']


In [32]:
recovery_df[
    [
        "payment_id",
        "customer_name",
        "amount",
        "base_recovery_probability",
        "selected_action",
        "recovered",
        "recovered_revenue"
    ]
]

,payment_id,customer_name,amount,base_recovery_probability,selected_action,recovered,recovered_revenue
0,pay_demo_101,Rahul,2499.0,70.24,offer_discount,True,2449.02
1,pay_demo_102,Priya,7999.0,67.97,offer_discount,True,7839.02
2,pay_demo_103,Arjun,1299.0,55.88,offer_discount,True,1273.02
3,pay_demo_104,Sneha,12499.0,80.04,offer_discount,True,12249.02
4,pay_demo_105,Kabir,499.0,30.25,offer_discount,True,489.02


In [33]:
total_failed_revenue = recovery_df[
    "amount"
].sum()

total_recovered_revenue = recovery_df[
    "recovered_revenue"
].sum()

recovered_transactions = recovery_df[
    "recovered"
].sum()

total_transactions = len(
    recovery_df
)

recovery_rate = (
    recovered_transactions /
    total_transactions
) * 100

revenue_recovery_rate = (
    total_recovered_revenue /
    total_failed_revenue
) * 100


In [34]:
print("==========================================")
print("             REVIVE AI")
print("       PERFORMANCE DASHBOARD")
print("==========================================")

print(
    f"Failed Revenue       : ₹"
    f"{total_failed_revenue:,.2f}"
)

print(
    f"Recovered Revenue    : ₹"
    f"{total_recovered_revenue:,.2f}"
)

print(
    f"Payments Recovered   : "
    f"{int(recovered_transactions)} / "
    f"{total_transactions}"
)

print(
    f"Recovery Rate        : "
    f"{recovery_rate:.2f}%"
)

print(
    f"Revenue Recovery     : "
    f"{revenue_recovery_rate:.2f}%"
)

print("==========================================")

             REVIVE AI
       PERFORMANCE DASHBOARD
Failed Revenue       : ₹24,795.00
Recovered Revenue    : ₹24,299.10
Payments Recovered   : 5 / 5
Recovery Rate        : 100.00%
Revenue Recovery     : 98.00%


In [35]:
!pip install -q gradio

In [36]:
import gradio as gr

In [42]:
# ============================================================
# REVIVE AI — DASHBOARD ENGINE
# ============================================================

def run_revive_dashboard(customer_name):

    # -----------------------------------------
    # Find selected payment
    # -----------------------------------------

    payment = next(
        p for p in demo_payments
        if p["customer_name"] == customer_name
    )

    # -----------------------------------------
    # Prepare ML input
    # -----------------------------------------

    model_input = {
        "amount": payment["amount"],
        "payment_method": payment["payment_method"],
        "failure_reason": payment["failure_reason"],
        "previous_successful_payments":
            payment["previous_successful_payments"],
        "previous_failed_payments":
            payment["previous_failed_payments"],
        "customer_age_days":
            payment["customer_age_days"],
        "attempt_number":
            payment["attempt_number"],
        "hour":
            payment["hour"]
    }

    payment_df = pd.DataFrame(
        [model_input]
    )

    # -----------------------------------------
    # ML prediction
    # -----------------------------------------

    base_probability = float(
        logistic_model.predict_proba(
            payment_df
        )[0][1]
    )

    # -----------------------------------------
    # Revenue optimizer
    # -----------------------------------------

    results, best_action = (
        optimize_recovery_action(
            payment,
            base_probability
        )
    )

    # -----------------------------------------
    # Agent decision
    # -----------------------------------------

    agent_decision = simulated_llm_agent(
        results
    )

    # -----------------------------------------
    # Execute
    # -----------------------------------------

    execution = execute_agent_decision(
        payment,
        agent_decision
    )

    # -----------------------------------------
    # Simulate customer response
    # -----------------------------------------

    customer_result = simulate_customer_outcome(
        payment,
        agent_decision["action"],
        results
    )

    # -----------------------------------------
    # Store transaction
    # -----------------------------------------

    event = {

        "timestamp":
            datetime.now().isoformat(),

        "payment_id":
            payment["payment_id"],

        "customer_name":
            payment["customer_name"],

        "amount":
            float(payment["amount"]),

        "failure_reason":
            payment["failure_reason"],

        "base_recovery_probability":
            round(
                base_probability * 100,
                2
            ),

        "selected_action":
            agent_decision["action"],

        "expected_net_revenue":
            round(
                float(
                    agent_decision[
                        "expected_net_revenue"
                    ]
                ),
                2
            ),

        "execution_status":
            execution["status"],

        "recovered":
            bool(
                customer_result["recovered"]
            ),

        "recovered_revenue":
            float(
                customer_result[
                    "recovered_revenue"
                ]
            )
    }

    # Don't duplicate the same transaction repeatedly
    existing_ids = [
        x["payment_id"]
        for x in recovery_log
    ]

    if payment["payment_id"] not in existing_ids:
        recovery_log.append(event)

    # -----------------------------------------
    # Strategy comparison table
    # -----------------------------------------

    strategy_df = pd.DataFrame(results)

    strategy_df["probability"] = (
        strategy_df["probability"] * 100
    ).round(2)

    strategy_df["expected_revenue"] = (
        strategy_df["expected_revenue"]
    ).round(2)

    strategy_df["cost"] = (
        strategy_df["cost"]
    ).round(2)

    strategy_df["expected_net_revenue"] = (
        strategy_df["expected_net_revenue"]
    ).round(2)

    strategy_df.columns = [
        "Action",
        "Recovery Probability %",
        "Expected Revenue ₹",
        "Cost ₹",
        "Expected Net Revenue ₹"
    ]

    # -----------------------------------------
    # Dashboard summary
    # -----------------------------------------

    status = (
        "🟢 PAYMENT RECOVERED"
        if customer_result["recovered"]
        else
        "🔴 PAYMENT NOT RECOVERED"
    )

    summary = f"""
# 🤖 REVIVE AI — Recovery Decision

### Transaction

| Field | Value |
|---|---|
| Customer | **{payment["customer_name"]}** |
| Payment ID | `{payment["payment_id"]}` |
| Amount | **₹{payment["amount"]:,.2f}** |
| Payment Method | **{payment["payment_method"]}** |
| Failure | **{payment["failure_reason"]}** |

---

### 🧠 AI Analysis

**Base Recovery Probability**

# {base_probability * 100:.2f}%

**Recommended Action**

# `{agent_decision["action"]}`

**Expected Net Revenue**

# ₹{agent_decision["expected_net_revenue"]:,.2f}

**AI Reasoning**

> {agent_decision["reason"]}

---

### {status}

**Recovered Revenue:** ₹{customer_result["recovered_revenue"]:,.2f}
"""

    # -----------------------------------------
    # KPI calculations
    # -----------------------------------------

    df = pd.DataFrame(
        recovery_log
    )

    if len(df) > 0:

        failed_revenue = df[
            "amount"
        ].sum()

        recovered_revenue = df[
            "recovered_revenue"
        ].sum()

        recovered_count = df[
            "recovered"
        ].sum()

        transaction_count = len(df)

        recovery_rate = (
            recovered_count /
            transaction_count
        ) * 100

    else:

        failed_revenue = 0
        recovered_revenue = 0
        recovery_rate = 0
        transaction_count = 0

    kpis = (
        f"₹{failed_revenue:,.0f}",
        f"₹{recovered_revenue:,.0f}",
        f"{recovery_rate:.1f}%",
        str(transaction_count)
    )

    return (
        summary,
        strategy_df,
        *kpis
    )


print("✅ REVIVE dashboard engine ready!")

✅ REVIVE dashboard engine ready!


In [38]:
# ============================================================
# REVIVE AI — INTERACTIVE DASHBOARD
# ============================================================

with gr.Blocks(
    title="REVIVE AI"
) as app:

    gr.Markdown(
        """
        # 💳 REVIVE AI
        ## Autonomous Revenue Recovery Agent

        **Predict → Optimize → Act → Recover**
        """
    )

    gr.Markdown(
        """
        REVIVE analyzes failed payments, predicts recovery
        probability, compares recovery strategies, and selects
        the strategy with the highest expected net revenue.
        """
    )

    # ========================================================
    # KPI ROW
    # ========================================================

    with gr.Row():

        failed_revenue = gr.Textbox(
            label="💸 Failed Revenue",
            value="₹0",
            interactive=False
        )

        recovered_revenue = gr.Textbox(
            label="💰 Recovered Revenue",
            value="₹0",
            interactive=False
        )

        recovery_rate = gr.Textbox(
            label="📈 Recovery Rate",
            value="0%",
            interactive=False
        )

        transactions = gr.Textbox(
            label="🤖 Transactions Analysed",
            value="0",
            interactive=False
        )

    gr.Markdown("---")

    # ========================================================
    # PAYMENT SELECTION
    # ========================================================

    with gr.Row():

        customer = gr.Dropdown(
            choices=[
                p["customer_name"]
                for p in demo_payments
            ],
            value=demo_payments[0]["customer_name"],
            label="Select Failed Payment"
        )

        analyze = gr.Button(
            "🚀 Run REVIVE AI",
            variant="primary"
        )

    # ========================================================
    # DECISION OUTPUT
    # ========================================================

    decision = gr.Markdown()

    # ========================================================
    # STRATEGY TABLE
    # ========================================================

    gr.Markdown(
        """
        ## 💰 Recovery Strategy Comparison
        """
    )

    strategy_table = gr.Dataframe(
        interactive=False
    )

    # ========================================================
    # BUTTON ACTION
    # ========================================================

    analyze.click(
        fn=run_revive_dashboard,

        inputs=[
            customer
        ],

        outputs=[
            decision,
            strategy_table,
            failed_revenue,
            recovered_revenue,
            recovery_rate,
            transactions
        ]
    )


app.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://222373513dfea087e8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [2]:
# ============================================================
# REVIVE AI — DEMO PAYMENT DATA
# ============================================================

demo_payments = [
    {
        "payment_id": "pay_demo_101",
        "customer_name": "Rahul",
        "amount": 2499,
        "payment_method": "upi",
        "failure_reason": "bank_timeout",
        "previous_successful_payments": 8,
        "previous_failed_payments": 1,
        "customer_age_days": 340,
        "attempt_number": 1,
        "hour": 14
    },

    {
        "payment_id": "pay_demo_102",
        "customer_name": "Priya",
        "amount": 7999,
        "payment_method": "card",
        "failure_reason": "card_declined",
        "previous_successful_payments": 12,
        "previous_failed_payments": 2,
        "customer_age_days": 520,
        "attempt_number": 1,
        "hour": 18
    },

    {
        "payment_id": "pay_demo_103",
        "customer_name": "Arjun",
        "amount": 1299,
        "payment_method": "upi",
        "failure_reason": "insufficient_balance",
        "previous_successful_payments": 2,
        "previous_failed_payments": 1,
        "customer_age_days": 90,
        "attempt_number": 1,
        "hour": 11
    },

    {
        "payment_id": "pay_demo_104",
        "customer_name": "Sneha",
        "amount": 12499,
        "payment_method": "netbanking",
        "failure_reason": "technical_error",
        "previous_successful_payments": 15,
        "previous_failed_payments": 1,
        "customer_age_days": 720,
        "attempt_number": 1,
        "hour": 20
    },

    {
        "payment_id": "pay_demo_105",
        "customer_name": "Kabir",
        "amount": 499,
        "payment_method": "card",
        "failure_reason": "authentication_failed",
        "previous_successful_payments": 0,
        "previous_failed_payments": 4,
        "customer_age_days": 12,
        "attempt_number": 3,
        "hour": 2
    }
]

print("✅ Demo payments restored!")
print(f"Total demo payments: {len(demo_payments)}")

for p in demo_payments:
    print(
        f"{p['payment_id']} | "
        f"{p['customer_name']} | "
        f"₹{p['amount']}"
    )

✅ Demo payments restored!
Total demo payments: 5
pay_demo_101 | Rahul | ₹2499
pay_demo_102 | Priya | ₹7999
pay_demo_103 | Arjun | ₹1299
pay_demo_104 | Sneha | ₹12499
pay_demo_105 | Kabir | ₹499


In [3]:
print("logistic_model:", "logistic_model" in globals())
print("demo_payments:", "demo_payments" in globals())
print("optimizer:", "optimize_recovery_action" in globals())
print("agent:", "simulated_llm_agent" in globals())
print("executor:", "execute_agent_decision" in globals())
print("outcome simulator:", "simulate_customer_outcome" in globals())
print("recovery log:", "recovery_log" in globals())

logistic_model: False
demo_payments: True
optimizer: False
agent: False
executor: False
outcome simulator: False
recovery log: False


In [4]:
print("demo_payments:", "demo_payments" in globals())
print("logistic_model:", "logistic_model" in globals())
print("optimizer:", "optimize_recovery_action" in globals())
print("agent:", "simulated_llm_agent" in globals())
print("executor:", "execute_agent_decision" in globals())
print("outcome simulator:", "simulate_customer_outcome" in globals())
print("recovery log:", "recovery_log" in globals())

demo_payments: True
logistic_model: False
optimizer: False
agent: False
executor: False
outcome simulator: False
recovery log: False


In [5]:
print("demo_payments:", "demo_payments" in globals())
print("logistic_model:", "logistic_model" in globals())
print("optimizer:", "optimize_recovery_action" in globals())
print("agent:", "simulated_llm_agent" in globals())
print("executor:", "execute_agent_decision" in globals())
print("outcome simulator:", "simulate_customer_outcome" in globals())

demo_payments: True
logistic_model: False
optimizer: False
agent: False
executor: False
outcome simulator: False


In [6]:
recovery_log = []

In [7]:
# ============================================================
# REVIVE AI — DEMO MODE
# ============================================================

DEMO_MODE = True

def simulate_customer_outcome(
    payment,
    selected_action,
    optimization_results
):

    selected_result = next(
        result
        for result in optimization_results
        if result["action"] == selected_action
    )

    probability = selected_result["probability"]

    # ----------------------------------------
    # Controlled demo outcome
    # ----------------------------------------

    if DEMO_MODE:

        # Strong recovery actions succeed
        recovered = probability >= 0.65

    else:

        recovered = (
            np.random.random() < probability
        )

    if recovered:

        recovered_amount = float(
            payment["amount"]
        )

        if selected_action == "offer_discount":
            recovered_amount *= 0.98

        print(
            f"🎉 Payment recovered: "
            f"₹{recovered_amount:.2f}"
        )

        return {
            "recovered": True,
            "recovered_revenue":
                round(recovered_amount, 2)
        }

    else:

        print("❌ Payment not recovered")

        return {
            "recovered": False,
            "recovered_revenue": 0
        }


print("🎬 REVIVE Demo Mode:", DEMO_MODE)

🎬 REVIVE Demo Mode: True


In [8]:
# ============================================================
# REVIVE AI — MERCHANT POLICY
# ============================================================

MERCHANT_POLICY = {

    "max_discount":
        0.02,

    "minimum_recovery_probability":
        0.15,

    "max_retry_attempts":
        2,

    "allow_discount":
        True,

    "allow_payment_link":
        True,

    "allow_message":
        True
}


print("✅ Merchant policy loaded")

print(
    f"Maximum discount: "
    f"{MERCHANT_POLICY['max_discount'] * 100}%"
)

print(
    f"Minimum recovery probability: "
    f"{MERCHANT_POLICY['minimum_recovery_probability'] * 100}%"
)

✅ Merchant policy loaded
Maximum discount: 2.0%
Minimum recovery probability: 15.0%


In [9]:
def apply_merchant_policy(
    payment,
    results
):

    filtered_results = []

    for result in results:

        action = result["action"]

        probability = (
            result["probability"]
        )

        # Minimum probability rule
        if (
            probability <
            MERCHANT_POLICY[
                "minimum_recovery_probability"
            ]
        ):
            continue

        # Discount permission
        if (
            action == "offer_discount"
            and not MERCHANT_POLICY[
                "allow_discount"
            ]
        ):
            continue

        # Payment link permission
        if (
            action == "generate_payment_link"
            and not MERCHANT_POLICY[
                "allow_payment_link"
            ]
        ):
            continue

        # Message permission
        if (
            action == "send_message"
            and not MERCHANT_POLICY[
                "allow_message"
            ]
        ):
            continue

        # Retry attempt limit
        if (
            action in [
                "retry_now",
                "retry_later"
            ]
            and payment["attempt_number"]
            >= MERCHANT_POLICY[
                "max_retry_attempts"
            ]
        ):
            continue

        filtered_results.append(result)

    return filtered_results

In [14]:
# ============================================================
# REVIVE AI — MASTER ML SETUP
# Run this whenever the Colab runtime resets
# ============================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

np.random.seed(42)

print("🔧 Building REVIVE ML system...")


# ============================================================
# 1. CREATE TRAINING DATA
# ============================================================

n = 10000

data = pd.DataFrame({

    "amount":
        np.random.randint(100, 20000, n),

    "payment_method":
        np.random.choice(
            ["upi", "card", "netbanking", "wallet"],
            n
        ),

    "failure_reason":
        np.random.choice(
            [
                "insufficient_balance",
                "bank_timeout",
                "card_declined",
                "technical_error",
                "authentication_failed"
            ],
            n
        ),

    "previous_successful_payments":
        np.random.randint(0, 20, n),

    "previous_failed_payments":
        np.random.randint(0, 6, n),

    "customer_age_days":
        np.random.randint(1, 1000, n),

    "attempt_number":
        np.random.randint(1, 4, n),

    "hour":
        np.random.randint(0, 24, n)
})


# ============================================================
# 2. CREATE SYNTHETIC RECOVERY OUTCOME
# ============================================================

def recovery_score(row):

    score = 0

    score += (
        row["previous_successful_payments"] * 0.08
    )

    score -= (
        row["previous_failed_payments"] * 0.10
    )

    if row["customer_age_days"] > 180:
        score += 0.20

    if row["failure_reason"] == "bank_timeout":
        score += 0.30

    elif row["failure_reason"] == "technical_error":
        score += 0.20

    elif row["failure_reason"] == "insufficient_balance":
        score += 0.10

    elif row["failure_reason"] == "card_declined":
        score -= 0.15

    elif row["failure_reason"] == "authentication_failed":
        score -= 0.25

    score -= (
        row["attempt_number"] - 1
    ) * 0.15

    if row["amount"] > 10000:
        score -= 0.10

    return 1 / (1 + np.exp(-score))


data["recovery_probability"] = data.apply(
    recovery_score,
    axis=1
)

data["recovered"] = np.random.binomial(
    1,
    data["recovery_probability"]
)


# ============================================================
# 3. FEATURES
# ============================================================

features = [
    "amount",
    "payment_method",
    "failure_reason",
    "previous_successful_payments",
    "previous_failed_payments",
    "customer_age_days",
    "attempt_number",
    "hour"
]

X = data[features]
y = data["recovered"]


categorical_features = [
    "payment_method",
    "failure_reason"
]

numeric_features = [
    "amount",
    "previous_successful_payments",
    "previous_failed_payments",
    "customer_age_days",
    "attempt_number",
    "hour"
]


# ============================================================
# 4. PREPROCESSOR
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[

        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),

        (
            "numeric",
            StandardScaler(),
            numeric_features
        )
    ]
)


# ============================================================
# 5. LOGISTIC REGRESSION MODEL
# ============================================================

logistic_model = Pipeline(
    steps=[

        (
            "preprocessor",
            preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)


# ============================================================
# 6. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ============================================================
# 7. TRAIN
# ============================================================

logistic_model.fit(
    X_train,
    y_train
)


# ============================================================
# 8. EVALUATE
# ============================================================

predictions = logistic_model.predict(
    X_test
)

probabilities = logistic_model.predict_proba(
    X_test
)[:, 1]

accuracy = accuracy_score(
    y_test,
    predictions
)

auc = roc_auc_score(
    y_test,
    probabilities
)


# ============================================================
# 9. SAVE MODEL
# ============================================================

joblib.dump(
    logistic_model,
    "revive_recovery_model.pkl"
)


print()
print("==========================================")
print("       ✅ REVIVE ML MODEL READY")
print("==========================================")
print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")
print(f"Accuracy         : {accuracy:.3f}")
print(f"ROC-AUC          : {auc:.3f}")
print("Model file       : revive_recovery_model.pkl")
print("==========================================")

🔧 Building REVIVE ML system...

       ✅ REVIVE ML MODEL READY
Training samples : 8000
Testing samples  : 2000
Accuracy         : 0.639
ROC-AUC          : 0.637
Model file       : revive_recovery_model.pkl


In [15]:
payment = demo_payments[0]

model_input = {
    "amount": payment["amount"],
    "payment_method": payment["payment_method"],
    "failure_reason": payment["failure_reason"],
    "previous_successful_payments":
        payment["previous_successful_payments"],
    "previous_failed_payments":
        payment["previous_failed_payments"],
    "customer_age_days":
        payment["customer_age_days"],
    "attempt_number":
        payment["attempt_number"],
    "hour":
        payment["hour"]
}

payment_df = pd.DataFrame(
    [model_input]
)

base_probability = float(
    logistic_model.predict_proba(
        payment_df
    )[0][1]
)

print(
    f"🧠 Rahul recovery probability: "
    f"{base_probability * 100:.2f}%"
)

🧠 Rahul recovery probability: 70.24%


In [17]:
# ============================================================
# REVIVE AI — MASTER OPTIMIZER + POLICY ENGINE
# ============================================================

# ------------------------------------------------------------
# 1. Action costs
# ------------------------------------------------------------

ACTION_COSTS = {
    "retry_now": 0,
    "retry_later": 0,
    "generate_payment_link": 0,
    "send_message": 2,
    "offer_discount": 0,
    "abandon_recovery": 0
}


# ------------------------------------------------------------
# 2. Estimate recovery probability for each action
# ------------------------------------------------------------

def estimate_action_probability(
    base_probability,
    action,
    failure_reason,
    previous_successful_payments
):

    probability = float(base_probability)

    if action == "retry_now":

        if failure_reason in [
            "bank_timeout",
            "technical_error"
        ]:
            probability += 0.10
        else:
            probability += 0.03

    elif action == "retry_later":

        if failure_reason in [
            "bank_timeout",
            "technical_error"
        ]:
            probability += 0.15
        else:
            probability += 0.05

    elif action == "generate_payment_link":

        probability += 0.12

    elif action == "send_message":

        probability += 0.10

        if previous_successful_payments >= 5:
            probability += 0.05

    elif action == "offer_discount":

        probability += 0.25

    elif action == "abandon_recovery":

        probability = 0

    return min(max(probability, 0), 0.99)


# ------------------------------------------------------------
# 3. Calculate expected net revenue
# ------------------------------------------------------------

def calculate_expected_value(
    payment,
    base_probability,
    action
):

    amount = float(payment["amount"])

    probability = estimate_action_probability(
        base_probability,
        action,
        payment["failure_reason"],
        payment["previous_successful_payments"]
    )

    # Discount action
    if action == "offer_discount":

        discount_percentage = 0.02

        customer_amount = (
            amount *
            (1 - discount_percentage)
        )

        expected_revenue = (
            probability *
            customer_amount
        )

        return {
            "action": action,
            "probability": probability,
            "expected_revenue": expected_revenue,
            "cost": amount * discount_percentage,
            "expected_net_revenue":
                expected_revenue
        }

    # Abandon
    if action == "abandon_recovery":

        return {
            "action": action,
            "probability": 0,
            "expected_revenue": 0,
            "cost": 0,
            "expected_net_revenue": 0
        }

    # Normal action
    cost = ACTION_COSTS[action]

    expected_revenue = (
        probability * amount
    )

    expected_net_revenue = (
        expected_revenue - cost
    )

    return {
        "action": action,
        "probability": probability,
        "expected_revenue": expected_revenue,
        "cost": cost,
        "expected_net_revenue":
            expected_net_revenue
    }


# ------------------------------------------------------------
# 4. Compare all strategies
# ------------------------------------------------------------

def optimize_recovery_action(
    payment,
    base_probability
):

    actions = [
        "retry_now",
        "retry_later",
        "generate_payment_link",
        "send_message",
        "offer_discount",
        "abandon_recovery"
    ]

    results = []

    for action in actions:

        result = calculate_expected_value(
            payment,
            base_probability,
            action
        )

        results.append(result)

    best_action = max(
        results,
        key=lambda x:
        x["expected_net_revenue"]
    )

    return results, best_action


# ============================================================
# 5. MERCHANT POLICY
# ============================================================

MERCHANT_POLICY = {

    "max_discount": 0.02,

    "minimum_recovery_probability": 0.15,

    "max_retry_attempts": 2,

    "allow_discount": True,

    "allow_payment_link": True,

    "allow_message": True
}


# ------------------------------------------------------------
# 6. Apply merchant policy
# ------------------------------------------------------------

def apply_merchant_policy(
    payment,
    results
):

    allowed_results = []

    for result in results:

        action = result["action"]
        probability = result["probability"]

        # Minimum probability
        if probability < MERCHANT_POLICY[
            "minimum_recovery_probability"
        ]:
            continue

        # Discount permission
        if (
            action == "offer_discount"
            and not MERCHANT_POLICY[
                "allow_discount"
            ]
        ):
            continue

        # Payment link permission
        if (
            action == "generate_payment_link"
            and not MERCHANT_POLICY[
                "allow_payment_link"
            ]
        ):
            continue

        # Message permission
        if (
            action == "send_message"
            and not MERCHANT_POLICY[
                "allow_message"
            ]
        ):
            continue

        # Retry limit
        if (
            action in [
                "retry_now",
                "retry_later"
            ]
            and payment["attempt_number"]
            >= MERCHANT_POLICY[
                "max_retry_attempts"
            ]
        ):
            continue

        allowed_results.append(result)

    return allowed_results


# ------------------------------------------------------------
# 7. Policy-aware AI decision
# ------------------------------------------------------------

def policy_aware_decision(
    payment,
    base_probability
):

    results, _ = optimize_recovery_action(
        payment,
        base_probability
    )

    allowed_results = apply_merchant_policy(
        payment,
        results
    )

    if len(allowed_results) == 0:

        return {
            "action": "abandon_recovery",
            "probability": 0,
            "expected_net_revenue": 0,
            "reason":
                "No recovery action satisfies "
                "merchant policy."
        }

    best = max(
        allowed_results,
        key=lambda x:
        x["expected_net_revenue"]
    )

    return {
        "action":
            best["action"],

        "probability":
            float(best["probability"]),

        "expected_net_revenue":
            float(
                best["expected_net_revenue"]
            ),

        "reason":
            (
                f"Selected {best['action']} "
                "because it provides the highest "
                "expected net revenue while "
                "respecting merchant policies."
            )
    }


print("==========================================")
print("✅ REVIVE OPTIMIZER READY")
print("✅ MERCHANT POLICY READY")
print("✅ POLICY-AWARE DECISION ENGINE READY")
print("==========================================")

✅ REVIVE OPTIMIZER READY
✅ MERCHANT POLICY READY
✅ POLICY-AWARE DECISION ENGINE READY


In [18]:
payment = demo_payments[0]

results, best_action = optimize_recovery_action(
    payment,
    base_probability
)

print("======================================")
print("REVIVE OPTIMIZER TEST")
print("======================================")

for r in results:

    print(
        f"{r['action']:25} | "
        f"{r['probability'] * 100:6.2f}% | "
        f"₹{r['expected_net_revenue']:,.2f}"
    )

print("\n🏆 Best action:")
print(best_action["action"])

REVIVE OPTIMIZER TEST
retry_now                 |  80.24% | ₹2,005.10
retry_later               |  85.24% | ₹2,130.05
generate_payment_link     |  82.24% | ₹2,055.08
send_message              |  85.24% | ₹2,128.05
offer_discount            |  95.24% | ₹2,332.36
abandon_recovery          |   0.00% | ₹0.00

🏆 Best action:
offer_discount


In [19]:
decision = policy_aware_decision(
    payment,
    base_probability
)

print("======================================")
print("🛡️ POLICY-AWARE REVIVE")
print("======================================")

print(
    "Action:",
    decision["action"]
)

print(
    "Probability:",
    f"{decision['probability'] * 100:.2f}%"
)

print(
    "Expected Net Revenue:",
    f"₹{decision['expected_net_revenue']:,.2f}"
)

print(
    "Reason:",
    decision["reason"]
)

🛡️ POLICY-AWARE REVIVE
Action: offer_discount
Probability: 95.24%
Expected Net Revenue: ₹2,332.36
Reason: Selected offer_discount because it provides the highest expected net revenue while respecting merchant policies.


In [20]:
MERCHANT_POLICY["allow_discount"] = False

decision_no_discount = policy_aware_decision(
    payment,
    base_probability
)

print(
    "Without discounts:",
    decision_no_discount["action"]
)

Without discounts: retry_later


In [21]:
MERCHANT_POLICY["allow_discount"] = True

decision_with_discount = policy_aware_decision(
    payment,
    base_probability
)

print(
    "With discounts:",
    decision_with_discount["action"]
)

With discounts: offer_discount


In [22]:
print("demo_payments:", "demo_payments" in globals())
print("logistic_model:", "logistic_model" in globals())

demo_payments: True
logistic_model: True


In [23]:
# ============================================================
# REVIVE AI — FINAL CONTROL CENTER
# ============================================================

import gradio as gr
import pandas as pd
import numpy as np
from datetime import datetime

# ------------------------------------------------------------
# GLOBALS
# ------------------------------------------------------------

if "recovery_log" not in globals():
    recovery_log = []


# ============================================================
# MERCHANT POLICY
# ============================================================

REVIVE_POLICY = {
    "max_discount": 0.02,
    "minimum_probability": 0.15,
    "max_retry_attempts": 2,
    "allow_discount": True,
    "allow_payment_link": True,
    "allow_message": True
}


# ============================================================
# AGENT
# ============================================================

def revive_agent(results):

    allowed = [
        r for r in results
        if r["allowed"]
    ]

    if not allowed:
        return {
            "action": "abandon_recovery",
            "probability": 0,
            "expected_net_revenue": 0,
            "reason":
                "No action satisfies merchant policy."
        }

    best = max(
        allowed,
        key=lambda x: x["expected_net_revenue"]
    )

    return {
        "action": best["action"],
        "probability": best["probability"],
        "expected_net_revenue":
            best["expected_net_revenue"],
        "reason":
            (
                f"{best['action']} provides the highest "
                f"expected net revenue among permitted "
                f"recovery actions."
            )
    }


# ============================================================
# ACTION EXECUTOR
# ============================================================

def execute_action(payment, action):

    if action == "retry_now":
        return "Retry initiated"

    elif action == "retry_later":
        return "Retry scheduled"

    elif action == "generate_payment_link":
        return "Payment link generated"

    elif action == "send_message":
        return "Recovery message sent"

    elif action == "offer_discount":
        return "Recovery discount offered"

    return "Recovery abandoned"


# ============================================================
# CUSTOMER SIMULATOR
# ============================================================

def customer_response(probability):

    # Deterministic demo mode
    if probability >= 0.65:
        return True

    return False


# ============================================================
# STRATEGY CALCULATOR
# ============================================================

def calculate_strategies(
    payment,
    base_probability
):

    actions = [
        "retry_now",
        "retry_later",
        "generate_payment_link",
        "send_message",
        "offer_discount",
        "abandon_recovery"
    ]

    results = []

    for action in actions:

        # ------------------------------------
        # Probability adjustment
        # ------------------------------------

        probability = base_probability

        if action == "retry_now":

            if payment["failure_reason"] in [
                "bank_timeout",
                "technical_error"
            ]:
                probability += 0.10
            else:
                probability += 0.03

        elif action == "retry_later":

            if payment["failure_reason"] in [
                "bank_timeout",
                "technical_error"
            ]:
                probability += 0.15
            else:
                probability += 0.05

        elif action == "generate_payment_link":

            probability += 0.12

        elif action == "send_message":

            probability += 0.10

            if payment[
                "previous_successful_payments"
            ] >= 5:
                probability += 0.05

        elif action == "offer_discount":

            probability += 0.25

        elif action == "abandon_recovery":

            probability = 0

        probability = min(
            max(probability, 0),
            0.99
        )

        # ------------------------------------
        # Revenue
        # ------------------------------------

        amount = float(
            payment["amount"]
        )

        if action == "offer_discount":

            discount = REVIVE_POLICY[
                "max_discount"
            ]

            customer_amount = (
                amount *
                (1 - discount)
            )

            expected_revenue = (
                probability *
                customer_amount
            )

            cost = (
                amount *
                discount
            )

        else:

            expected_revenue = (
                probability *
                amount
            )

            cost = 2 if (
                action == "send_message"
            ) else 0

        net_revenue = (
            expected_revenue -
            cost
        )

        # ------------------------------------
        # Policy
        # ------------------------------------

        allowed = True

        if probability < REVIVE_POLICY[
            "minimum_probability"
        ]:
            allowed = False

        if (
            action == "offer_discount"
            and not REVIVE_POLICY[
                "allow_discount"
            ]
        ):
            allowed = False

        if (
            action == "generate_payment_link"
            and not REVIVE_POLICY[
                "allow_payment_link"
            ]
        ):
            allowed = False

        if (
            action == "send_message"
            and not REVIVE_POLICY[
                "allow_message"
            ]
        ):
            allowed = False

        if (
            action in [
                "retry_now",
                "retry_later"
            ]
            and payment["attempt_number"]
            >= REVIVE_POLICY[
                "max_retry_attempts"
            ]
        ):
            allowed = False

        results.append({

            "action": action,

            "probability":
                probability,

            "expected_revenue":
                expected_revenue,

            "cost":
                cost,

            "expected_net_revenue":
                net_revenue,

            "allowed":
                allowed
        })

    return results


# ============================================================
# MAIN REVIVE ENGINE
# ============================================================

def run_revive(
    customer_name,
    max_discount,
    minimum_probability,
    allow_discount
):

    # Update merchant policy
    REVIVE_POLICY[
        "max_discount"
    ] = max_discount / 100

    REVIVE_POLICY[
        "minimum_probability"
    ] = minimum_probability / 100

    REVIVE_POLICY[
        "allow_discount"
    ] = allow_discount

    # ----------------------------------------
    # Find payment
    # ----------------------------------------

    payment = next(
        p for p in demo_payments
        if p["customer_name"] == customer_name
    )

    # ----------------------------------------
    # ML input
    # ----------------------------------------

    model_input = {
        "amount":
            payment["amount"],

        "payment_method":
            payment["payment_method"],

        "failure_reason":
            payment["failure_reason"],

        "previous_successful_payments":
            payment[
                "previous_successful_payments"
            ],

        "previous_failed_payments":
            payment[
                "previous_failed_payments"
            ],

        "customer_age_days":
            payment["customer_age_days"],

        "attempt_number":
            payment["attempt_number"],

        "hour":
            payment["hour"]
    }

    payment_df = pd.DataFrame(
        [model_input]
    )

    # ----------------------------------------
    # ML prediction
    # ----------------------------------------

    base_probability = float(
        logistic_model.predict_proba(
            payment_df
        )[0][1]
    )

    # ----------------------------------------
    # Strategy optimization
    # ----------------------------------------

    results = calculate_strategies(
        payment,
        base_probability
    )

    # ----------------------------------------
    # Agent decision
    # ----------------------------------------

    decision = revive_agent(
        results
    )

    # ----------------------------------------
    # Execute
    # ----------------------------------------

    execution_status = execute_action(
        payment,
        decision["action"]
    )

    # ----------------------------------------
    # Customer response
    # ----------------------------------------

    recovered = customer_response(
        decision["probability"]
    )

    if recovered:

        if decision["action"] == "offer_discount":

            recovered_revenue = (
                payment["amount"] *
                (1 - REVIVE_POLICY[
                    "max_discount"
                ])
            )

        else:

            recovered_revenue = payment[
                "amount"
            ]

    else:

        recovered_revenue = 0

    # ----------------------------------------
    # Audit record
    # ----------------------------------------

    event = {

        "timestamp":
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            ),

        "payment_id":
            payment["payment_id"],

        "customer":
            payment["customer_name"],

        "amount":
            payment["amount"],

        "failure":
            payment["failure_reason"],

        "recovery_probability":
            round(
                base_probability * 100,
                2
            ),

        "selected_action":
            decision["action"],

        "recovered":
            recovered,

        "recovered_revenue":
            round(
                recovered_revenue,
                2
            )
    }

    # Replace existing record for same payment
    global recovery_log

    recovery_log = [
        x for x in recovery_log
        if x["payment_id"]
        != payment["payment_id"]
    ]

    recovery_log.append(event)

    # ----------------------------------------
    # Strategy table
    # ----------------------------------------

    strategy_df = pd.DataFrame(
        results
    )

    strategy_df = strategy_df[
        [
            "action",
            "probability",
            "expected_revenue",
            "cost",
            "expected_net_revenue",
            "allowed"
        ]
    ].copy()

    strategy_df[
        "probability"
    ] = (
        strategy_df["probability"] * 100
    ).round(2)

    strategy_df[
        "expected_revenue"
    ] = strategy_df[
        "expected_revenue"
    ].round(2)

    strategy_df["cost"] = (
        strategy_df["cost"]
    ).round(2)

    strategy_df[
        "expected_net_revenue"
    ] = strategy_df[
        "expected_net_revenue"
    ].round(2)

    strategy_df.columns = [
        "Action",
        "Recovery %",
        "Expected Revenue ₹",
        "Cost ₹",
        "Net Revenue ₹",
        "Policy Allowed"
    ]

    # ----------------------------------------
    # Current totals
    # ----------------------------------------

    log_df = pd.DataFrame(
        recovery_log
    )

    failed_revenue = (
        log_df["amount"].sum()
        if len(log_df)
        else 0
    )

    recovered_revenue = (
        log_df[
            "recovered_revenue"
        ].sum()
        if len(log_df)
        else 0
    )

    recovery_rate = (
        log_df["recovered"].mean() * 100
        if len(log_df)
        else 0
    )

    # ----------------------------------------
    # UI summary
    # ----------------------------------------

    status = (
        "🟢 PAYMENT RECOVERED"
        if recovered
        else
        "🔴 PAYMENT NOT RECOVERED"
    )

    summary = f"""
# 🤖 REVIVE AI

## Autonomous Revenue Recovery Agent

---

### 💳 PAYMENT

| Field | Value |
|---|---|
| Customer | **{payment["customer_name"]}** |
| Payment ID | `{payment["payment_id"]}` |
| Amount | **₹{payment["amount"]:,.2f}** |
| Method | **{payment["payment_method"]}** |
| Failure | **{payment["failure_reason"]}** |

---

### 🧠 AI ANALYSIS

**Base Recovery Probability**

# {base_probability * 100:.2f}%

**AI Selected Action**

# `{decision["action"]}`

**Expected Net Revenue**

# ₹{decision["expected_net_revenue"]:,.2f}

**Execution**

`{execution_status}`

---

### 🤖 AI REASONING

> {decision["reason"]}

---

## {status}

**Recovered Revenue:** ₹{recovered_revenue:,.2f}
"""

    return (
        summary,
        strategy_df,
        failed_revenue,
        recovered_revenue,
        recovery_rate,
        len(log_df)
    )


# ============================================================
# GRADIO UI
# ============================================================

with gr.Blocks(
    title="REVIVE AI"
) as app:

    gr.Markdown(
        """
# 💳 REVIVE AI

### Autonomous Revenue Recovery Agent

**Predict → Optimize → Govern → Act → Recover**

> AI-powered recovery for failed payments.
"""
    )

    # --------------------------------------------------------
    # KPI CARDS
    # --------------------------------------------------------

    with gr.Row():

        failed_box = gr.Number(
            label="💸 Failed Revenue",
            value=0,
            interactive=False
        )

        recovered_box = gr.Number(
            label="💰 Recovered Revenue",
            value=0,
            interactive=False
        )

        rate_box = gr.Number(
            label="📈 Recovery Rate %",
            value=0,
            interactive=False
        )

        transaction_box = gr.Number(
            label="🤖 AI Transactions",
            value=0,
            interactive=False
        )

    gr.Markdown("---")

    # --------------------------------------------------------
    # CONTROLS
    # --------------------------------------------------------

    with gr.Row():

        customer_dropdown = gr.Dropdown(
            choices=[
                p["customer_name"]
                for p in demo_payments
            ],
            value=demo_payments[0][
                "customer_name"
            ],
            label="💳 Failed Payment"
        )

        max_discount_slider = gr.Slider(
            minimum=0,
            maximum=10,
            value=2,
            step=1,
            label="Maximum Discount %"
        )

        min_probability_slider = gr.Slider(
            minimum=0,
            maximum=100,
            value=15,
            step=5,
            label="Minimum Recovery Probability %"
        )

        allow_discount_checkbox = gr.Checkbox(
            value=True,
            label="Allow AI Discount"
        )

    run_button = gr.Button(
        "🚀 RUN REVIVE AI",
        variant="primary"
    )

    gr.Markdown("---")

    # --------------------------------------------------------
    # DECISION
    # --------------------------------------------------------

    decision_output = gr.Markdown()

    # --------------------------------------------------------
    # STRATEGIES
    # --------------------------------------------------------

    gr.Markdown(
        """
## 💰 Strategy Evaluation

REVIVE evaluates multiple recovery actions and selects
the highest-value policy-compliant strategy.
"""
    )

    strategy_output = gr.Dataframe(
        interactive=False
    )

    # --------------------------------------------------------
    # EVENT
    # --------------------------------------------------------

    run_button.click(
        fn=run_revive,

        inputs=[
            customer_dropdown,
            max_discount_slider,
            min_probability_slider,
            allow_discount_checkbox
        ],

        outputs=[
            decision_output,
            strategy_output,
            failed_box,
            recovered_box,
            rate_box,
            transaction_box
        ]
    )


# ============================================================
# LAUNCH
# ============================================================

print("🚀 Starting REVIVE AI Control Center...")

app.launch(
    share=True,
    debug=True
)

🚀 Starting REVIVE AI Control Center...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://30ec6e7113f8176a28.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://30ec6e7113f8176a28.gradio.live


In [24]:
import gradio as gr

with gr.Blocks() as test_app:
    gr.Markdown("# REVIVE AI TEST")
    gr.Markdown("Dashboard server is working.")

test_app.launch(
    share=False,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [1]:
# Stop anything currently using Gradio's default port
!fuser -k 7860/tcp 2>/dev/null || true

print("✅ Gradio port cleaned")


✅ Gradio port cleaned


In [2]:
import gradio as gr

with gr.Blocks() as test_app:
    gr.Markdown("# 🚀 REVIVE AI TEST")
    gr.Markdown("If you can see this page, the public Gradio link works.")

test_app.launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b0018226c9e609474a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:
# ============================================================
# REVIVE AI — RESTORE DEMO PAYMENTS
# ============================================================

demo_payments = [
    {
        "payment_id": "pay_demo_101",
        "customer_name": "Rahul",
        "amount": 2499,
        "payment_method": "upi",
        "failure_reason": "bank_timeout",
        "previous_successful_payments": 8,
        "previous_failed_payments": 1,
        "customer_age_days": 340,
        "attempt_number": 1,
        "hour": 14
    },
    {
        "payment_id": "pay_demo_102",
        "customer_name": "Priya",
        "amount": 7999,
        "payment_method": "card",
        "failure_reason": "card_declined",
        "previous_successful_payments": 12,
        "previous_failed_payments": 2,
        "customer_age_days": 520,
        "attempt_number": 1,
        "hour": 18
    },
    {
        "payment_id": "pay_demo_103",
        "customer_name": "Arjun",
        "amount": 1299,
        "payment_method": "upi",
        "failure_reason": "insufficient_balance",
        "previous_successful_payments": 2,
        "previous_failed_payments": 1,
        "customer_age_days": 90,
        "attempt_number": 1,
        "hour": 11
    },
    {
        "payment_id": "pay_demo_104",
        "customer_name": "Sneha",
        "amount": 12499,
        "payment_method": "netbanking",
        "failure_reason": "technical_error",
        "previous_successful_payments": 15,
        "previous_failed_payments": 1,
        "customer_age_days": 720,
        "attempt_number": 1,
        "hour": 20
    },
    {
        "payment_id": "pay_demo_105",
        "customer_name": "Kabir",
        "amount": 499,
        "payment_method": "card",
        "failure_reason": "authentication_failed",
        "previous_successful_payments": 0,
        "previous_failed_payments": 4,
        "customer_age_days": 12,
        "attempt_number": 3,
        "hour": 2
    }
]

print("✅ Demo payments restored")
print("Total payments:", len(demo_payments))

✅ Demo payments restored
Total payments: 5


In [6]:
print("demo_payments:", "demo_payments" in globals())
print("logistic_model:", "logistic_model" in globals())
print("optimizer:", "optimize_recovery_action" in globals())

demo_payments: True
logistic_model: False
optimizer: False


In [7]:
# ============================================================
# REVIVE AI — STABLE BUILDATHON DASHBOARD
# ============================================================

import gradio as gr
import pandas as pd
import numpy as np
from datetime import datetime


# ============================================================
# 1. DEMO PAYMENT DATA
# ============================================================

REVIVE_PAYMENTS = [
    {
        "payment_id": "pay_demo_101",
        "customer_name": "Rahul",
        "amount": 2499,
        "payment_method": "upi",
        "failure_reason": "bank_timeout",
        "previous_successful_payments": 8,
        "previous_failed_payments": 1,
        "customer_age_days": 340,
        "attempt_number": 1,
        "hour": 14
    },
    {
        "payment_id": "pay_demo_102",
        "customer_name": "Priya",
        "amount": 7999,
        "payment_method": "card",
        "failure_reason": "card_declined",
        "previous_successful_payments": 12,
        "previous_failed_payments": 2,
        "customer_age_days": 520,
        "attempt_number": 1,
        "hour": 18
    },
    {
        "payment_id": "pay_demo_103",
        "customer_name": "Arjun",
        "amount": 1299,
        "payment_method": "upi",
        "failure_reason": "insufficient_balance",
        "previous_successful_payments": 2,
        "previous_failed_payments": 1,
        "customer_age_days": 90,
        "attempt_number": 1,
        "hour": 11
    },
    {
        "payment_id": "pay_demo_104",
        "customer_name": "Sneha",
        "amount": 12499,
        "payment_method": "netbanking",
        "failure_reason": "technical_error",
        "previous_successful_payments": 15,
        "previous_failed_payments": 1,
        "customer_age_days": 720,
        "attempt_number": 1,
        "hour": 20
    },
    {
        "payment_id": "pay_demo_105",
        "customer_name": "Kabir",
        "amount": 499,
        "payment_method": "card",
        "failure_reason": "authentication_failed",
        "previous_successful_payments": 0,
        "previous_failed_payments": 4,
        "customer_age_days": 12,
        "attempt_number": 3,
        "hour": 2
    }
]


# ============================================================
# 2. POLICY
# ============================================================

REVIVE_POLICY = {
    "minimum_probability": 0.15,
    "max_retry_attempts": 2,
    "allow_discount": True
}


# ============================================================
# 3. RECOVERY PROBABILITY
# ============================================================

def get_recovery_probability(payment):

    # --------------------------------------------------------
    # Use trained ML model when available
    # --------------------------------------------------------

    if "logistic_model" in globals():

        try:

            model_input = {
                "amount": payment["amount"],
                "payment_method": payment["payment_method"],
                "failure_reason": payment["failure_reason"],
                "previous_successful_payments":
                    payment["previous_successful_payments"],
                "previous_failed_payments":
                    payment["previous_failed_payments"],
                "customer_age_days":
                    payment["customer_age_days"],
                "attempt_number":
                    payment["attempt_number"],
                "hour":
                    payment["hour"]
            }

            df = pd.DataFrame([model_input])

            probability = float(
                logistic_model.predict_proba(df)[0][1]
            )

            return probability, "ML Model"

        except Exception:
            pass


    # --------------------------------------------------------
    # Fallback heuristic
    # --------------------------------------------------------

    score = 0.50

    score += min(
        payment["previous_successful_payments"] * 0.025,
        0.20
    )

    score -= min(
        payment["previous_failed_payments"] * 0.04,
        0.16
    )

    if payment["failure_reason"] == "bank_timeout":
        score += 0.15

    elif payment["failure_reason"] == "technical_error":
        score += 0.12

    elif payment["failure_reason"] == "insufficient_balance":
        score -= 0.05

    elif payment["failure_reason"] == "card_declined":
        score -= 0.10

    elif payment["failure_reason"] == "authentication_failed":
        score -= 0.20

    if payment["customer_age_days"] > 180:
        score += 0.05

    score -= (
        payment["attempt_number"] - 1
    ) * 0.08

    return min(max(score, 0.01), 0.99), "Fallback Model"


# ============================================================
# 4. STRATEGY OPTIMIZER
# ============================================================

def optimize_payment(
    payment,
    base_probability,
    max_discount,
    allow_discount
):

    strategies = []

    actions = [
        "retry_now",
        "retry_later",
        "generate_payment_link",
        "send_message"
    ]

    if allow_discount:
        actions.append("offer_discount")

    actions.append("abandon_recovery")


    for action in actions:

        probability = base_probability

        # --------------------------------------------
        # Action impact
        # --------------------------------------------

        if action == "retry_now":

            if payment["failure_reason"] in [
                "bank_timeout",
                "technical_error"
            ]:
                probability += 0.10
            else:
                probability += 0.03


        elif action == "retry_later":

            if payment["failure_reason"] in [
                "bank_timeout",
                "technical_error"
            ]:
                probability += 0.15
            else:
                probability += 0.05


        elif action == "generate_payment_link":

            probability += 0.12


        elif action == "send_message":

            probability += 0.10

            if payment[
                "previous_successful_payments"
            ] >= 5:
                probability += 0.05


        elif action == "offer_discount":

            probability += 0.25


        elif action == "abandon_recovery":

            probability = 0


        probability = min(
            max(probability, 0),
            0.99
        )


        # --------------------------------------------
        # Revenue calculation
        # --------------------------------------------

        amount = float(payment["amount"])


        if action == "offer_discount":

            customer_amount = (
                amount * (1 - max_discount)
            )

            expected_revenue = (
                probability * customer_amount
            )

            discount_cost = (
                amount * max_discount
            )

            net_revenue = expected_revenue


        elif action == "abandon_recovery":

            expected_revenue = 0
            discount_cost = 0
            net_revenue = 0


        else:

            expected_revenue = (
                probability * amount
            )

            discount_cost = (
                2 if action == "send_message"
                else 0
            )

            net_revenue = (
                expected_revenue -
                discount_cost
            )


        # --------------------------------------------
        # Policy
        # --------------------------------------------

        allowed = True

        if probability < REVIVE_POLICY[
            "minimum_probability"
        ]:
            allowed = False


        if (
            action in [
                "retry_now",
                "retry_later"
            ]
            and payment["attempt_number"]
            >= REVIVE_POLICY[
                "max_retry_attempts"
            ]
        ):
            allowed = False


        strategies.append({
            "action": action,
            "probability": probability,
            "expected_revenue": expected_revenue,
            "cost": discount_cost,
            "net_revenue": net_revenue,
            "allowed": allowed
        })


    return strategies


# ============================================================
# 5. MAIN REVIVE ENGINE
# ============================================================

def run_revive_dashboard(
    customer_name,
    max_discount,
    minimum_probability,
    allow_discount
):

    REVIVE_POLICY["minimum_probability"] = (
        minimum_probability / 100
    )

    REVIVE_POLICY["allow_discount"] = (
        allow_discount
    )

    payment = next(
        p for p in REVIVE_PAYMENTS
        if p["customer_name"] == customer_name
    )


    # --------------------------------------------------------
    # ML
    # --------------------------------------------------------

    probability, model_source = (
        get_recovery_probability(payment)
    )


    # --------------------------------------------------------
    # Optimization
    # --------------------------------------------------------

    strategies = optimize_payment(
        payment,
        probability,
        max_discount / 100,
        allow_discount
    )


    # --------------------------------------------------------
    # AI chooses best permitted action
    # --------------------------------------------------------

    allowed = [
        x for x in strategies
        if x["allowed"]
    ]


    if allowed:

        decision = max(
            allowed,
            key=lambda x: x["net_revenue"]
        )

    else:

        decision = {
            "action": "abandon_recovery",
            "probability": 0,
            "expected_revenue": 0,
            "cost": 0,
            "net_revenue": 0,
            "allowed": True
        }


    # --------------------------------------------------------
    # Deterministic demo result
    # --------------------------------------------------------

    recovered = (
        decision["probability"] >= 0.65
        and decision["action"]
        != "abandon_recovery"
    )


    if recovered:

        if decision["action"] == "offer_discount":

            recovered_revenue = (
                payment["amount"]
                * (1 - max_discount / 100)
            )

        else:

            recovered_revenue = (
                payment["amount"]
            )

    else:

        recovered_revenue = 0


    # --------------------------------------------------------
    # Strategy table
    # --------------------------------------------------------

    table = pd.DataFrame(strategies)

    table = table[
        [
            "action",
            "probability",
            "expected_revenue",
            "cost",
            "net_revenue",
            "allowed"
        ]
    ].copy()


    table["probability"] = (
        table["probability"] * 100
    ).round(2)


    table["expected_revenue"] = (
        table["expected_revenue"]
    ).round(2)


    table["cost"] = (
        table["cost"]
    ).round(2)


    table["net_revenue"] = (
        table["net_revenue"]
    ).round(2)


    table.columns = [
        "Action",
        "Recovery %",
        "Expected Revenue ₹",
        "Cost ₹",
        "Net Revenue ₹",
        "Policy Allowed"
    ]


    # --------------------------------------------------------
    # Result
    # --------------------------------------------------------

    status = (
        "🟢 PAYMENT RECOVERED"
        if recovered
        else
        "🔴 PAYMENT NOT RECOVERED"
    )


    summary = f"""
# 💳 REVIVE AI

## Autonomous Revenue Recovery Agent

---

### PAYMENT

**Customer:** {payment["customer_name"]}

**Payment ID:** `{payment["payment_id"]}`

**Amount:** ₹{payment["amount"]:,.2f}

**Method:** {payment["payment_method"]}

**Failure:** `{payment["failure_reason"]}`

---

### 🧠 AI ANALYSIS

**Prediction Source:** `{model_source}`

**Base Recovery Probability:**

# {probability * 100:.2f}%

---

### 🤖 REVIVE DECISION

**Selected Action:**

# `{decision["action"]}`

**Expected Net Revenue:**

# ₹{decision["net_revenue"]:,.2f}

**Reason:**

> REVIVE selected **{decision["action"]}** because it
> provides the highest expected net revenue among
> policy-approved strategies.

---

### {status}

**Recovered Revenue:** ₹{recovered_revenue:,.2f}

---

### 🔐 Merchant Guardrails

Maximum Discount: **{max_discount:.0f}%**

Minimum Recovery Probability:
**{minimum_probability:.0f}%**

Discount Allowed:
**{"YES" if allow_discount else "NO"}**
"""


    return (
        summary,
        table,
        payment["amount"],
        recovered_revenue,
        recovered * 100,
        1
    )


# ============================================================
# 6. BUILD UI
# ============================================================

with gr.Blocks(
    title="REVIVE AI"
) as revive_app:

    gr.Markdown(
        """
# 💳 REVIVE AI

## Autonomous Revenue Recovery Agent

### **Predict → Optimize → Govern → Act → Recover**
"""
    )


    # --------------------------------------------------------
    # KPI ROW
    # --------------------------------------------------------

    with gr.Row():

        failed_box = gr.Number(
            label="💸 Failed Revenue",
            value=0,
            interactive=False
        )

        recovered_box = gr.Number(
            label="💰 Recovered Revenue",
            value=0,
            interactive=False
        )

        rate_box = gr.Number(
            label="📈 Recovery Rate %",
            value=0,
            interactive=False
        )

        action_box = gr.Number(
            label="🤖 AI Transactions",
            value=0,
            interactive=False
        )


    gr.Markdown("---")


    # --------------------------------------------------------
    # CONTROLS
    # --------------------------------------------------------

    customer_dropdown = gr.Dropdown(
        choices=[
            p["customer_name"]
            for p in REVIVE_PAYMENTS
        ],
        value="Rahul",
        label="💳 Select Failed Payment"
    )


    with gr.Row():

        discount_slider = gr.Slider(
            minimum=0,
            maximum=10,
            value=2,
            step=1,
            label="Maximum Discount %"
        )

        probability_slider = gr.Slider(
            minimum=0,
            maximum=100,
            value=15,
            step=5,
            label="Minimum Recovery Probability %"
        )

        discount_checkbox = gr.Checkbox(
            value=True,
            label="Allow AI Discount"
        )


    run_button = gr.Button(
        "🚀 RUN REVIVE AI",
        variant="primary"
    )


    gr.Markdown("---")


    # --------------------------------------------------------
    # DECISION
    # --------------------------------------------------------

    decision_output = gr.Markdown()


    # --------------------------------------------------------
    # STRATEGY TABLE
    # --------------------------------------------------------

    gr.Markdown(
        """
## 💰 Recovery Strategy Evaluation
"""
    )

    strategy_output = gr.Dataframe(
        interactive=False
    )


    # --------------------------------------------------------
    # EVENT
    # --------------------------------------------------------

    run_button.click(
        fn=run_revive_dashboard,

        inputs=[
            customer_dropdown,
            discount_slider,
            probability_slider,
            discount_checkbox
        ],

        outputs=[
            decision_output,
            strategy_output,
            failed_box,
            recovered_box,
            rate_box,
            action_box
        ]
    )


print("✅ REVIVE AI dashboard created!")

✅ REVIVE AI dashboard created!


In [8]:
revive_app.launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://569b3db0d9f4c506dd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
# ============================================================
# REVIVE AI — STEP 12: AI DECISION TRACE
# ============================================================

def generate_decision_trace(
    payment,
    base_probability,
    strategies,
    decision,
    max_discount,
    minimum_probability,
    allow_discount
):

    # Sort strategies by net revenue
    ranked = sorted(
        strategies,
        key=lambda x: x["net_revenue"],
        reverse=True
    )

    lines = []

    lines.append(
        "## 🤖 REVIVE AI — DECISION TRACE"
    )

    lines.append("---")

    lines.append(
        "### 1️⃣ PAYMENT ANALYSIS"
    )

    lines.append(
        f"- Customer: **{payment['customer_name']}**"
    )

    lines.append(
        f"- Payment: `{payment['payment_id']}`"
    )

    lines.append(
        f"- Amount: **₹{payment['amount']:,.2f}**"
    )

    lines.append(
        f"- Method: **{payment['payment_method']}**"
    )

    lines.append(
        f"- Failure: `{payment['failure_reason']}`"
    )

    lines.append("---")

    lines.append(
        "### 2️⃣ CUSTOMER BEHAVIOUR"
    )

    lines.append(
        f"- Successful payments: "
        f"**{payment['previous_successful_payments']}**"
    )

    lines.append(
        f"- Previous failures: "
        f"**{payment['previous_failed_payments']}**"
    )

    lines.append(
        f"- Customer age: "
        f"**{payment['customer_age_days']} days**"
    )

    lines.append(
        f"- Current attempt: "
        f"**{payment['attempt_number']}**"
    )

    lines.append("---")

    lines.append(
        "### 3️⃣ 🧠 ML PREDICTION"
    )

    lines.append(
        f"Base recovery probability: "
        f"**{base_probability * 100:.2f}%**"
    )

    lines.append("---")

    lines.append(
        "### 4️⃣ 💰 STRATEGY EVALUATION"
    )

    for i, strategy in enumerate(
        ranked,
        start=1
    ):

        allowed = (
            "✅ Allowed"
            if strategy["allowed"]
            else
            "🚫 Blocked"
        )

        lines.append(
            f"{i}. **{strategy['action']}** — "
            f"{strategy['probability'] * 100:.2f}% "
            f"recovery — "
            f"₹{strategy['net_revenue']:,.2f} "
            f"net revenue — "
            f"{allowed}"
        )

    lines.append("---")

    lines.append(
        "### 5️⃣ 🛡️ MERCHANT GUARDRAILS"
    )

    lines.append(
        f"- Maximum discount: "
        f"**{max_discount:.0f}%**"
    )

    lines.append(
        f"- Minimum recovery probability: "
        f"**{minimum_probability:.0f}%**"
    )

    lines.append(
        f"- Discount allowed: "
        f"**{'YES' if allow_discount else 'NO'}**"
    )

    lines.append("---")

    lines.append(
        "### 6️⃣ 🤖 AUTONOMOUS DECISION"
    )

    lines.append(
        f"### Selected action: "
        f"**`{decision['action']}`**"
    )

    lines.append(
        f"Expected net revenue: "
        f"**₹{decision['net_revenue']:,.2f}**"
    )

    # Find selected strategy
    selected = next(
        (
            x for x in strategies
            if x["action"] == decision["action"]
        ),
        None
    )

    if selected:

        lines.append(
            f"Recovery probability after action: "
            f"**{selected['probability'] * 100:.2f}%**"
        )

    lines.append("---")

    lines.append(
        "### 7️⃣ 💡 WHY REVIVE CHOSE THIS"
    )

    lines.append(
        f"> REVIVE evaluated "
        f"**{len(strategies)} recovery strategies** "
        f"and selected **{decision['action']}** "
        f"because it produced the highest expected "
        f"net revenue among actions permitted by "
        f"merchant policy."
    )

    return "\n".join(lines)


print("✅ AI Decision Trace ready!")

✅ AI Decision Trace ready!


In [10]:
# ============================================================
# TEST DECISION TRACE
# ============================================================

payment = REVIVE_PAYMENTS[0]

base_probability, model_source = (
    get_recovery_probability(payment)
)

strategies = optimize_payment(
    payment,
    base_probability,
    2,
    True
)

allowed = [
    x for x in strategies
    if x["allowed"]
]

decision = max(
    allowed,
    key=lambda x: x["net_revenue"]
)

trace = generate_decision_trace(
    payment,
    base_probability,
    strategies,
    decision,
    2,
    15,
    True
)

print(trace)

## 🤖 REVIVE AI — DECISION TRACE
---
### 1️⃣ PAYMENT ANALYSIS
- Customer: **Rahul**
- Payment: `pay_demo_101`
- Amount: **₹2,499.00**
- Method: **upi**
- Failure: `bank_timeout`
---
### 2️⃣ CUSTOMER BEHAVIOUR
- Successful payments: **8**
- Previous failures: **1**
- Customer age: **340 days**
- Current attempt: **1**
---
### 3️⃣ 🧠 ML PREDICTION
Base recovery probability: **86.00%**
---
### 4️⃣ 💰 STRATEGY EVALUATION
1. **retry_later** — 99.00% recovery — ₹2,474.01 net revenue — ✅ Allowed
2. **send_message** — 99.00% recovery — ₹2,472.01 net revenue — ✅ Allowed
3. **generate_payment_link** — 98.00% recovery — ₹2,449.02 net revenue — ✅ Allowed
4. **retry_now** — 96.00% recovery — ₹2,399.04 net revenue — ✅ Allowed
5. **abandon_recovery** — 0.00% recovery — ₹0.00 net revenue — 🚫 Blocked
6. **offer_discount** — 99.00% recovery — ₹-2,474.01 net revenue — ✅ Allowed
---
### 5️⃣ 🛡️ MERCHANT GUARDRAILS
- Maximum discount: **2%**
- Minimum recovery probability: **15%**
- Discount allowed: **YES**
-

In [11]:
# ============================================================
# REVIVE AI — PORTFOLIO ANALYTICS
# ============================================================

def generate_portfolio_analytics():

    records = []

    for payment in REVIVE_PAYMENTS:

        probability, _ = (
            get_recovery_probability(payment)
        )

        strategies = optimize_payment(
            payment,
            probability,
            0.02,
            True
        )

        allowed = [
            x for x in strategies
            if x["allowed"]
        ]

        if allowed:

            decision = max(
                allowed,
                key=lambda x: x["net_revenue"]
            )

        else:

            decision = {
                "action": "abandon_recovery",
                "probability": 0,
                "net_revenue": 0
            }

        recovered = (
            decision["probability"] >= 0.65
            and decision["action"]
            != "abandon_recovery"
        )

        if recovered:

            if decision["action"] == "offer_discount":

                recovered_revenue = (
                    payment["amount"] * 0.98
                )

            else:

                recovered_revenue = (
                    payment["amount"]
                )

        else:

            recovered_revenue = 0

        records.append({

            "payment_id":
                payment["payment_id"],

            "customer":
                payment["customer_name"],

            "amount":
                payment["amount"],

            "base_probability":
                probability,

            "selected_action":
                decision["action"],

            "recovered":
                recovered,

            "recovered_revenue":
                recovered_revenue
        })

    return pd.DataFrame(records)


portfolio_df = generate_portfolio_analytics()

portfolio_df

,payment_id,customer,amount,base_probability,selected_action,recovered,recovered_revenue
0,pay_demo_101,Rahul,2499,0.86,retry_later,True,2499.00
1,pay_demo_102,Priya,7999,0.57,offer_discount,True,7839.02
2,pay_demo_103,Arjun,1299,0.46,offer_discount,True,1273.02
3,pay_demo_104,Sneha,12499,0.83,retry_later,True,12499.00
4,pay_demo_105,Kabir,499,0.01,offer_discount,False,0.00


In [12]:
# ============================================================
# REVIVE AI — PORTFOLIO ANALYTICS
# ============================================================

def generate_portfolio_analytics():

    records = []

    for payment in REVIVE_PAYMENTS:

        probability, _ = (
            get_recovery_probability(payment)
        )

        strategies = optimize_payment(
            payment,
            probability,
            0.02,
            True
        )

        allowed = [
            x for x in strategies
            if x["allowed"]
        ]

        if allowed:

            decision = max(
                allowed,
                key=lambda x: x["net_revenue"]
            )

        else:

            decision = {
                "action": "abandon_recovery",
                "probability": 0,
                "net_revenue": 0
            }

        recovered = (
            decision["probability"] >= 0.65
            and decision["action"]
            != "abandon_recovery"
        )

        if recovered:

            if decision["action"] == "offer_discount":

                recovered_revenue = (
                    payment["amount"] * 0.98
                )

            else:

                recovered_revenue = (
                    payment["amount"]
                )

        else:

            recovered_revenue = 0

        records.append({

            "payment_id":
                payment["payment_id"],

            "customer":
                payment["customer_name"],

            "amount":
                payment["amount"],

            "base_probability":
                probability,

            "selected_action":
                decision["action"],

            "recovered":
                recovered,

            "recovered_revenue":
                recovered_revenue
        })

    return pd.DataFrame(records)


portfolio_df = generate_portfolio_analytics()

portfolio_df

,payment_id,customer,amount,base_probability,selected_action,recovered,recovered_revenue
0,pay_demo_101,Rahul,2499,0.86,retry_later,True,2499.00
1,pay_demo_102,Priya,7999,0.57,offer_discount,True,7839.02
2,pay_demo_103,Arjun,1299,0.46,offer_discount,True,1273.02
3,pay_demo_104,Sneha,12499,0.83,retry_later,True,12499.00
4,pay_demo_105,Kabir,499,0.01,offer_discount,False,0.00


In [13]:
# ============================================================
# REVIVE AI — BUSINESS KPIs
# ============================================================

total_failed_revenue = (
    portfolio_df["amount"].sum()
)

total_recovered_revenue = (
    portfolio_df["recovered_revenue"].sum()
)

recovery_rate = (
    portfolio_df["recovered"].mean()
    * 100
)

revenue_recovery_percentage = (
    total_recovered_revenue /
    total_failed_revenue *
    100
    if total_failed_revenue > 0
    else 0
)

print("==========================================")
print("        💰 REVIVE BUSINESS IMPACT")
print("==========================================")

print(
    f"Failed Revenue: "
    f"₹{total_failed_revenue:,.2f}"
)

print(
    f"Recovered Revenue: "
    f"₹{total_recovered_revenue:,.2f}"
)

print(
    f"Payment Recovery Rate: "
    f"{recovery_rate:.2f}%"
)

print(
    f"Revenue Recovery: "
    f"{revenue_recovery_percentage:.2f}%"
)

print("==========================================")

        💰 REVIVE BUSINESS IMPACT
Failed Revenue: ₹24,795.00
Recovered Revenue: ₹24,110.04
Payment Recovery Rate: 80.00%
Revenue Recovery: 97.24%


In [14]:
action_summary = (
    portfolio_df
    .groupby("selected_action")
    .agg(
        payments=("payment_id", "count"),
        recovered=("recovered", "sum"),
        recovered_revenue=(
            "recovered_revenue",
            "sum"
        )
    )
    .reset_index()
)

action_summary

,selected_action,payments,recovered,recovered_revenue
0,offer_discount,3,2,9112.04
1,retry_later,2,2,14998.00


In [15]:
# ============================================================
# REVIVE AI — STEP 12 FINAL DASHBOARD
# ============================================================

with gr.Blocks(
    title="REVIVE AI — Revenue Recovery"
) as revive_v2:

    gr.Markdown("""
# 💳 REVIVE AI

## Autonomous Revenue Recovery Agent

### Predict → Optimize → Govern → Act → Recover

**AI-powered revenue recovery for failed payments**
""")

    gr.Markdown("---")

    # ========================================================
    # PAYMENT
    # ========================================================

    gr.Markdown("## 💳 Failed Payment")

    customer = gr.Dropdown(
        choices=[
            p["customer_name"]
            for p in REVIVE_PAYMENTS
        ],
        value="Rahul",
        label="Select Failed Payment"
    )

    # ========================================================
    # POLICY
    # ========================================================

    gr.Markdown("## 🛡️ Merchant Guardrails")

    with gr.Row():

        discount = gr.Slider(
            minimum=0,
            maximum=10,
            value=2,
            step=1,
            label="Maximum Discount %"
        )

        minimum_prob = gr.Slider(
            minimum=0,
            maximum=100,
            value=15,
            step=5,
            label="Minimum Recovery Probability %"
        )

        allow_discount = gr.Checkbox(
            value=True,
            label="Allow AI Discount"
        )

    run = gr.Button(
        "🚀 RUN REVIVE AI",
        variant="primary"
    )

    gr.Markdown("---")

    # ========================================================
    # DECISION TRACE
    # ========================================================

    gr.Markdown(
        "## 🤖 AI Decision Trace"
    )

    trace_output = gr.Markdown()

    # ========================================================
    # STRATEGIES
    # ========================================================

    gr.Markdown(
        "## 💰 Strategy Evaluation"
    )

    strategy_output = gr.Dataframe(
        interactive=False
    )

    # ========================================================
    # BUSINESS IMPACT
    # ========================================================

    gr.Markdown(
        "## 📊 Revenue Impact"
    )

    with gr.Row():

        failed_revenue = gr.Number(
            label="💸 Failed Revenue",
            interactive=False
        )

        recovered_revenue = gr.Number(
            label="💰 Recovered Revenue",
            interactive=False
        )

        recovery_percentage = gr.Number(
            label="📈 Revenue Recovery %",
            interactive=False
        )

        payment_rate = gr.Number(
            label="🟢 Payment Recovery %",
            interactive=False
        )

    # ========================================================
    # FUNCTION
    # ========================================================

    def run_v2(
        customer_name,
        max_discount,
        min_probability,
        allow_discount
    ):

        payment = next(
            p for p in REVIVE_PAYMENTS
            if p["customer_name"]
            == customer_name
        )

        probability, model_source = (
            get_recovery_probability(
                payment
            )
        )

        strategies = optimize_payment(
            payment,
            probability,
            max_discount / 100,
            allow_discount
        )

        allowed = [
            x for x in strategies
            if x["allowed"]
        ]

        if allowed:

            decision = max(
                allowed,
                key=lambda x:
                x["net_revenue"]
            )

        else:

            decision = {
                "action":
                    "abandon_recovery",
                "probability": 0,
                "net_revenue": 0
            }

        trace = generate_decision_trace(
            payment,
            probability,
            strategies,
            decision,
            max_discount,
            min_probability,
            allow_discount
        )

        table = pd.DataFrame(
            strategies
        )

        table["probability"] = (
            table["probability"] * 100
        ).round(2)

        table["expected_revenue"] = (
            table["expected_revenue"]
        ).round(2)

        table["cost"] = (
            table["cost"]
        ).round(2)

        table["net_revenue"] = (
            table["net_revenue"]
        ).round(2)

        table.columns = [
            "Action",
            "Recovery %",
            "Expected Revenue ₹",
            "Cost ₹",
            "Net Revenue ₹",
            "Policy Allowed"
        ]

        # --------------------------------------------
        # Simulated result
        # --------------------------------------------

        recovered = (
            decision["probability"] >= 0.65
            and decision["action"]
            != "abandon_recovery"
        )

        if recovered:

            if decision["action"] == "offer_discount":

                recovered_amount = (
                    payment["amount"]
                    * (1 - max_discount / 100)
                )

            else:

                recovered_amount = (
                    payment["amount"]
                )

        else:

            recovered_amount = 0

        # --------------------------------------------
        # Portfolio calculation
        # --------------------------------------------

        portfolio = generate_portfolio_analytics()

        failed_total = (
            portfolio["amount"].sum()
        )

        recovered_total = (
            portfolio["recovered_revenue"].sum()
        )

        payment_recovery = (
            portfolio["recovered"].mean()
            * 100
        )

        revenue_recovery = (
            recovered_total /
            failed_total *
            100
            if failed_total > 0
            else 0
        )

        return (
            trace,
            table,
            failed_total,
            recovered_total,
            revenue_recovery,
            payment_recovery
        )

    # ========================================================
    # EVENT
    # ========================================================

    run.click(
        fn=run_v2,

        inputs=[
            customer,
            discount,
            minimum_prob,
            allow_discount
        ],

        outputs=[
            trace_output,
            strategy_output,
            failed_revenue,
            recovered_revenue,
            recovery_percentage,
            payment_rate
        ]
    )


print(
    "✅ REVIVE AI Step 12 dashboard created!"
)

✅ REVIVE AI Step 12 dashboard created!


In [16]:
revive_v2.launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b1f7460f1bfec6056e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [17]:
# ============================================================
# REVIVE AI — STEP 13: ACTION EXECUTION TOOLS
# ============================================================

from datetime import datetime


def retry_payment(payment):
    """
    Simulates a payment retry.
    """

    return {
        "status": "EXECUTED",
        "action": "retry_now",
        "message": (
            f"Payment retry initiated for "
            f"{payment['payment_id']}"
        ),
        "timestamp":
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )
    }


def schedule_retry(payment):
    """
    Simulates scheduling a retry.
    """

    return {
        "status": "EXECUTED",
        "action": "retry_later",
        "message": (
            f"Retry scheduled for customer "
            f"{payment['customer_name']}"
        ),
        "timestamp":
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )
    }


def generate_payment_link(payment):
    """
    Simulates generation of a payment link.
    """

    fake_link = (
        "https://pay.revive.ai/"
        + payment["payment_id"]
    )

    return {
        "status": "EXECUTED",
        "action": "generate_payment_link",
        "message": (
            "Payment link generated"
        ),
        "payment_link": fake_link,
        "timestamp":
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )
    }


def send_recovery_message(payment):
    """
    Simulates sending a recovery message.
    """

    return {
        "status": "EXECUTED",
        "action": "send_message",
        "message": (
            f"Recovery message sent to "
            f"{payment['customer_name']}"
        ),
        "timestamp":
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )
    }


def offer_recovery_discount(
    payment,
    discount_percentage
):
    """
    Simulates offering a recovery discount.
    """

    original_amount = float(
        payment["amount"]
    )

    discount_amount = (
        original_amount *
        discount_percentage
    )

    final_amount = (
        original_amount -
        discount_amount
    )

    return {
        "status": "EXECUTED",
        "action": "offer_discount",
        "message": (
            "Recovery discount offered"
        ),
        "original_amount":
            round(original_amount, 2),
        "discount":
            round(discount_amount, 2),
        "customer_pays":
            round(final_amount, 2),
        "timestamp":
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )
    }


def abandon_recovery(payment):
    """
    Safely stops recovery attempts.
    """

    return {
        "status": "STOPPED",
        "action": "abandon_recovery",
        "message": (
            "Recovery abandoned safely"
        ),
        "timestamp":
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )
    }


print("✅ REVIVE action tools loaded!")

✅ REVIVE action tools loaded!


In [18]:
# ============================================================
# REVIVE AI — AUTONOMOUS ACTION EXECUTOR
# ============================================================

def execute_recovery_action(
    payment,
    action,
    discount_percentage=0.02
):

    print()
    print("==========================================")
    print("🤖 REVIVE AGENT — EXECUTING")
    print("==========================================")
    print(
        f"Customer: {payment['customer_name']}"
    )
    print(
        f"Payment: {payment['payment_id']}"
    )
    print(
        f"Selected action: {action}"
    )
    print("------------------------------------------")

    if action == "retry_now":

        result = retry_payment(payment)

    elif action == "retry_later":

        result = schedule_retry(payment)

    elif action == "generate_payment_link":

        result = generate_payment_link(
            payment
        )

    elif action == "send_message":

        result = send_recovery_message(
            payment
        )

    elif action == "offer_discount":

        result = offer_recovery_discount(
            payment,
            discount_percentage
        )

    elif action == "abandon_recovery":

        result = abandon_recovery(
            payment
        )

    else:

        result = {
            "status": "ERROR",
            "action": action,
            "message": (
                "Unknown recovery action"
            ),
            "timestamp":
                datetime.now().strftime(
                    "%Y-%m-%d %H:%M:%S"
                )
        }

    print(
        "Status:",
        result["status"]
    )

    print(
        "Message:",
        result["message"]
    )

    return result


print("✅ Autonomous executor ready!")

✅ Autonomous executor ready!


In [19]:
payment = REVIVE_PAYMENTS[0]

execution = execute_recovery_action(
    payment,
    "offer_discount",
    0.02
)

print()
print("Execution result:")
print(execution)


🤖 REVIVE AGENT — EXECUTING
Customer: Rahul
Payment: pay_demo_101
Selected action: offer_discount
------------------------------------------
Status: EXECUTED
Message: Recovery discount offered

Execution result:
{'status': 'EXECUTED', 'action': 'offer_discount', 'message': 'Recovery discount offered', 'original_amount': 2499.0, 'discount': 49.98, 'customer_pays': 2449.02, 'timestamp': '2026-09-05 18:47:04'}


In [20]:
# ============================================================
# CUSTOMER OUTCOME ENGINE
# ============================================================

def evaluate_customer_outcome(
    payment,
    execution,
    probability
):

    action = execution["action"]

    # Demo simulation
    recovered = (
        probability >= 0.65
        and action != "abandon_recovery"
    )

    if recovered:

        if action == "offer_discount":

            revenue = execution[
                "customer_pays"
            ]

        else:

            revenue = payment[
                "amount"
            ]

        status = "PAYMENT_RECOVERED"

    else:

        revenue = 0

        status = "PAYMENT_NOT_RECOVERED"

    return {
        "status": status,
        "recovered": recovered,
        "recovered_revenue":
            round(float(revenue), 2),
        "timestamp":
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )
    }


print("✅ Customer outcome engine ready!")

✅ Customer outcome engine ready!


In [21]:
# ============================================================
# COMPLETE REVIVE AGENT LOOP
# ============================================================

payment = REVIVE_PAYMENTS[0]

probability, model_source = (
    get_recovery_probability(payment)
)

strategies = optimize_payment(
    payment,
    probability,
    0.02,
    True
)

allowed = [
    x for x in strategies
    if x["allowed"]
]

decision = max(
    allowed,
    key=lambda x: x["net_revenue"]
)

execution = execute_recovery_action(
    payment,
    decision["action"],
    0.02
)

outcome = evaluate_customer_outcome(
    payment,
    execution,
    decision["probability"]
)

print()
print("==========================================")
print("🎯 REVIVE COMPLETE AGENT LOOP")
print("==========================================")

print(
    "ML probability:",
    f"{probability * 100:.2f}%"
)

print(
    "AI decision:",
    decision["action"]
)

print(
    "Expected net revenue:",
    f"₹{decision['net_revenue']:,.2f}"
)

print(
    "Execution:",
    execution["status"]
)

print(
    "Customer outcome:",
    outcome["status"]
)

print(
    "Recovered revenue:",
    f"₹{outcome['recovered_revenue']:,.2f}"
)


🤖 REVIVE AGENT — EXECUTING
Customer: Rahul
Payment: pay_demo_101
Selected action: retry_later
------------------------------------------
Status: EXECUTED
Message: Retry scheduled for customer Rahul

🎯 REVIVE COMPLETE AGENT LOOP
ML probability: 86.00%
AI decision: retry_later
Expected net revenue: ₹2,474.01
Execution: EXECUTED
Customer outcome: PAYMENT_RECOVERED
Recovered revenue: ₹2,499.00


In [22]:
# ============================================================
# REVIVE AI — AGENT ACTIVITY LOG
# ============================================================

agent_activity_log = []


def log_agent_activity(
    payment,
    probability,
    decision,
    execution,
    outcome
):

    event = {

        "timestamp":
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            ),

        "payment_id":
            payment["payment_id"],

        "customer":
            payment["customer_name"],

        "amount":
            payment["amount"],

        "failure_reason":
            payment["failure_reason"],

        "recovery_probability":
            round(
                probability * 100,
                2
            ),

        "ai_action":
            decision["action"],

        "expected_net_revenue":
            round(
                decision["net_revenue"],
                2
            ),

        "execution_status":
            execution["status"],

        "customer_outcome":
            outcome["status"],

        "recovered_revenue":
            outcome[
                "recovered_revenue"
            ]
    }

    agent_activity_log.append(
        event
    )

    return event


event = log_agent_activity(
    payment,
    probability,
    decision,
    execution,
    outcome
)

print("✅ Agent activity recorded")
print()
print(event)

✅ Agent activity recorded

{'timestamp': '2026-09-05 18:48:01', 'payment_id': 'pay_demo_101', 'customer': 'Rahul', 'amount': 2499, 'failure_reason': 'bank_timeout', 'recovery_probability': 86.0, 'ai_action': 'retry_later', 'expected_net_revenue': 2474.01, 'execution_status': 'EXECUTED', 'customer_outcome': 'PAYMENT_RECOVERED', 'recovered_revenue': 2499.0}


In [23]:
agent_log_df = pd.DataFrame(
    agent_activity_log
)

agent_log_df

,timestamp,payment_id,customer,amount,failure_reason,recovery_probability,ai_action,expected_net_revenue,execution_status,customer_outcome,recovered_revenue
0,2026-09-05 18:48:01,pay_demo_101,Rahul,2499,bank_timeout,86.0,retry_later,2474.01,EXECUTED,PAYMENT_RECOVERED,2499.0


In [24]:
import random
import uuid
from datetime import datetime

CUSTOMER_NAMES = [
    "Rahul", "Priya", "Arjun", "Sneha",
    "Kabir", "Meera", "Aman", "Isha"
]

PAYMENT_METHODS = [
    "upi",
    "card",
    "netbanking",
    "wallet"
]

FAILURE_REASONS = [
    "bank_timeout",
    "technical_error",
    "insufficient_balance",
    "card_declined",
    "authentication_failed"
]


def generate_failed_payment():

    payment = {
        "payment_id":
            "pay_" + uuid.uuid4().hex[:10],

        "customer_name":
            random.choice(CUSTOMER_NAMES),

        "amount":
            random.choice([
                499,
                999,
                1499,
                2499,
                4999,
                7999,
                12499
            ]),

        "payment_method":
            random.choice(PAYMENT_METHODS),

        "failure_reason":
            random.choice(FAILURE_REASONS),

        "previous_successful_payments":
            random.randint(0, 15),

        "previous_failed_payments":
            random.randint(0, 4),

        "customer_age_days":
            random.randint(10, 900),

        "attempt_number":
            random.randint(1, 3),

        "hour":
            random.randint(0, 23)
    }

    return payment


print("✅ Failed payment simulator ready!")

✅ Failed payment simulator ready!


In [25]:
def run_full_revive_agent(
    payment,
    max_discount=0.02,
    allow_discount=True
):

    # -----------------------------------------
    # 1. ML prediction
    # -----------------------------------------

    base_probability, model_source = (
        get_recovery_probability(payment)
    )

    # -----------------------------------------
    # 2. Strategy evaluation
    # -----------------------------------------

    strategies = optimize_payment(
        payment,
        base_probability,
        max_discount,
        allow_discount
    )

    allowed = [
        s for s in strategies
        if s["allowed"]
    ]

    # -----------------------------------------
    # 3. Agent decision
    # -----------------------------------------

    if allowed:

        decision = max(
            allowed,
            key=lambda x:
                x["net_revenue"]
        )

    else:

        decision = {
            "action":
                "abandon_recovery",

            "probability":
                0,

            "net_revenue":
                0
        }

    # -----------------------------------------
    # 4. Execute action
    # -----------------------------------------

    execution = execute_recovery_action(
        payment,
        decision["action"],
        max_discount
    )

    # -----------------------------------------
    # 5. Customer outcome
    # -----------------------------------------

    outcome = evaluate_customer_outcome(
        payment,
        execution,
        decision["probability"]
    )

    # -----------------------------------------
    # 6. Audit log
    # -----------------------------------------

    event = log_agent_activity(
        payment,
        base_probability,
        decision,
        execution,
        outcome
    )

    return {
        "payment":
            payment,

        "model_source":
            model_source,

        "base_probability":
            base_probability,

        "strategies":
            strategies,

        "decision":
            decision,

        "execution":
            execution,

        "outcome":
            outcome,

        "audit":
            event
    }


print("✅ Full REVIVE autonomous pipeline ready!")

✅ Full REVIVE autonomous pipeline ready!


In [26]:
test_payment = generate_failed_payment()

result = run_full_revive_agent(
    test_payment
)

print("\nNEW FAILED PAYMENT")
print("===================")

print(
    test_payment
)

print("\nREVIVE RESULT")
print("===================")

print(
    "Decision:",
    result["decision"]["action"]
)

print(
    "Recovered:",
    result["outcome"]["recovered"]
)

print(
    "Recovered Revenue:",
    result["outcome"]["recovered_revenue"]
)


🤖 REVIVE AGENT — EXECUTING
Customer: Isha
Payment: pay_78b445d9e4
Selected action: offer_discount
------------------------------------------
Status: EXECUTED
Message: Recovery discount offered

NEW FAILED PAYMENT
{'payment_id': 'pay_78b445d9e4', 'customer_name': 'Isha', 'amount': 999, 'payment_method': 'wallet', 'failure_reason': 'authentication_failed', 'previous_successful_payments': 1, 'previous_failed_payments': 1, 'customer_age_days': 633, 'attempt_number': 2, 'hour': 8}

REVIVE RESULT
Decision: offer_discount
Recovered: False
Recovered Revenue: 0.0


In [27]:
import gradio as gr
import pandas as pd

current_payment = None


def simulate_and_analyze(
    max_discount,
    allow_discount
):

    global current_payment

    # -----------------------------------------
    # Generate payment
    # -----------------------------------------

    current_payment = generate_failed_payment()

    # -----------------------------------------
    # Run agent
    # -----------------------------------------

    result = run_full_revive_agent(
        current_payment,
        max_discount / 100,
        allow_discount
    )

    payment = result["payment"]
    decision = result["decision"]
    outcome = result["outcome"]

    # -----------------------------------------
    # Decision trace
    # -----------------------------------------

    trace = generate_decision_trace(
        payment,
        result["base_probability"],
        result["strategies"],
        decision,
        max_discount,
        REVIVE_POLICY[
            "minimum_probability"
        ] * 100,
        allow_discount
    )

    # -----------------------------------------
    # Strategy table
    # -----------------------------------------

    strategy_df = pd.DataFrame(
        result["strategies"]
    )

    strategy_df[
        "probability"
    ] = (
        strategy_df[
            "probability"
        ] * 100
    ).round(2)

    strategy_df[
        "expected_revenue"
    ] = strategy_df[
        "expected_revenue"
    ].round(2)

    strategy_df[
        "cost"
    ] = strategy_df[
        "cost"
    ].round(2)

    strategy_df[
        "net_revenue"
    ] = strategy_df[
        "net_revenue"
    ].round(2)

    strategy_df.columns = [
        "Action",
        "Recovery %",
        "Expected Revenue ₹",
        "Cost ₹",
        "Net Revenue ₹",
        "Allowed"
    ]

    # -----------------------------------------
    # Payment summary
    # -----------------------------------------

    payment_summary = f"""
## 🚨 New Failed Payment Detected

**Payment ID:** `{payment["payment_id"]}`

**Customer:** {payment["customer_name"]}

**Amount:** ₹{payment["amount"]:,.2f}

**Method:** {payment["payment_method"]}

**Failure:** `{payment["failure_reason"]}`

**Previous Successful Payments:** {payment["previous_successful_payments"]}

**Previous Failed Payments:** {payment["previous_failed_payments"]}

**Attempt Number:** {payment["attempt_number"]}
"""

    # -----------------------------------------
    # Outcome summary
    # -----------------------------------------

    outcome_text = f"""
## ⚡ Agent Execution

**Selected Action:** `{decision["action"]}`

**Execution Status:** `{result["execution"]["status"]}`

**Customer Outcome:** `{outcome["status"]}`

### 💰 Recovered Revenue

# ₹{outcome["recovered_revenue"]:,.2f}
"""

    # -----------------------------------------
    # Audit dataframe
    # -----------------------------------------

    audit_df = pd.DataFrame(
        agent_activity_log
    )

    # -----------------------------------------
    # KPI calculations
    # -----------------------------------------

    failed_total = (
        audit_df["amount"].sum()
        if len(audit_df)
        else 0
    )

    recovered_total = (
        audit_df[
            "recovered_revenue"
        ].sum()
        if len(audit_df)
        else 0
    )

    recovery_rate = (
        (
            audit_df[
                "customer_outcome"
            ]
            == "PAYMENT_RECOVERED"
        ).mean() * 100
        if len(audit_df)
        else 0
    )

    return (
        payment_summary,
        trace,
        strategy_df,
        outcome_text,
        audit_df.tail(10),
        failed_total,
        recovered_total,
        recovery_rate,
        len(audit_df)
    )


with gr.Blocks(
    title="REVIVE AI — Live Demo"
) as revive_final:

    gr.Markdown("""
# 💳 REVIVE AI

## Autonomous Revenue Recovery Agent

### Predict → Optimize → Govern → Act → Recover

**Live Buildathon Demo**
""")

    # -----------------------------------------
    # KPI ROW
    # -----------------------------------------

    with gr.Row():

        failed_kpi = gr.Number(
            label="💸 Failed Revenue",
            value=0,
            interactive=False
        )

        recovered_kpi = gr.Number(
            label="💰 Recovered Revenue",
            value=0,
            interactive=False
        )

        recovery_kpi = gr.Number(
            label="📈 Recovery Rate %",
            value=0,
            interactive=False
        )

        actions_kpi = gr.Number(
            label="🤖 AI Actions",
            value=0,
            interactive=False
        )

    gr.Markdown("---")

    # -----------------------------------------
    # CONTROLS
    # -----------------------------------------

    gr.Markdown(
        "## 🛡️ Merchant Guardrails"
    )

    with gr.Row():

        discount_slider = gr.Slider(
            minimum=0,
            maximum=10,
            value=2,
            step=1,
            label="Maximum Discount %"
        )

        discount_toggle = gr.Checkbox(
            value=True,
            label="Allow AI Discount"
        )

    simulate_button = gr.Button(
        "🚨 SIMULATE NEW FAILED PAYMENT",
        variant="primary"
    )

    gr.Markdown("---")

    # -----------------------------------------
    # PAYMENT
    # -----------------------------------------

    payment_output = gr.Markdown()

    # -----------------------------------------
    # TRACE
    # -----------------------------------------

    gr.Markdown(
        "## 🤖 AI Decision Trace"
    )

    trace_output = gr.Markdown()

    # -----------------------------------------
    # STRATEGIES
    # -----------------------------------------

    gr.Markdown(
        "## 💰 Strategy Evaluation"
    )

    strategy_output = gr.Dataframe(
        interactive=False
    )

    # -----------------------------------------
    # OUTCOME
    # -----------------------------------------

    outcome_output = gr.Markdown()

    # -----------------------------------------
    # AUDIT LOG
    # -----------------------------------------

    gr.Markdown(
        "## 📋 Agent Audit Trail"
    )

    audit_output = gr.Dataframe(
        interactive=False
    )

    # -----------------------------------------
    # EVENT
    # -----------------------------------------

    simulate_button.click(
        fn=simulate_and_analyze,

        inputs=[
            discount_slider,
            discount_toggle
        ],

        outputs=[
            payment_output,
            trace_output,
            strategy_output,
            outcome_output,
            audit_output,
            failed_kpi,
            recovered_kpi,
            recovery_kpi,
            actions_kpi
        ]
    )


print(
    "✅ REVIVE final buildathon dashboard created!"
)

✅ REVIVE final buildathon dashboard created!


In [28]:
revive_final.launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://671a92200fffdb8861.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [29]:
# ============================================================
# REVIVE — MODEL PERFORMANCE SUMMARY
# ============================================================

print("""
==========================================
       REVIVE AI MODEL PERFORMANCE
==========================================
Model: Logistic Regression

Purpose:
Predict probability that a failed payment
can be successfully recovered.

Features:
• Payment amount
• Payment method
• Failure reason
• Previous successful payments
• Previous failed payments
• Customer age
• Attempt number
• Time of payment

Evaluation:
• Accuracy
• Precision
• Recall
• F1-score
• ROC-AUC

Important:
The ML model predicts recovery probability.
The revenue optimizer decides WHICH ACTION
should be taken.
==========================================
""")


       REVIVE AI MODEL PERFORMANCE
Model: Logistic Regression

Purpose:
Predict probability that a failed payment
can be successfully recovered.

Features:
• Payment amount
• Payment method
• Failure reason
• Previous successful payments
• Previous failed payments
• Customer age
• Attempt number
• Time of payment

Evaluation:
• Accuracy
• Precision
• Recall
• F1-score
• ROC-AUC

Important:
The ML model predicts recovery probability.
The revenue optimizer decides WHICH ACTION
should be taken.



In [30]:
README = """
# REVIVE AI

## Autonomous Revenue Recovery Agent

REVIVE AI is an agentic revenue recovery system designed
to recover failed payments while maximizing expected net
revenue and respecting merchant-defined policies.

## Problem

Failed payments create significant revenue leakage.
Traditional systems often retry payments using fixed rules.

REVIVE instead combines:

1. Machine learning
2. Revenue optimization
3. Merchant guardrails
4. Autonomous action execution
5. Explainable AI
6. Revenue analytics

## Architecture

Failed Payment
      ↓
ML Recovery Prediction
      ↓
Recovery Strategy Optimization
      ↓
Merchant Policy Engine
      ↓
AI Agent
      ↓
Action Execution
      ↓
Customer Outcome
      ↓
Revenue Analytics

## ML Model

The prototype uses Logistic Regression to predict the
probability that a failed payment can be recovered.

Features include:

- Payment amount
- Payment method
- Failure reason
- Previous successful payments
- Previous failed payments
- Customer age
- Attempt number
- Payment time

## Agent Actions

REVIVE can evaluate:

- Retry now
- Retry later
- Generate payment link
- Send recovery message
- Offer discount
- Abandon recovery

## Revenue Optimization

Each action is evaluated using:

Expected Revenue =
Recovery Probability × Recoverable Amount

The agent chooses the policy-approved action with the
highest expected net revenue.

## Merchant Guardrails

Merchants can control:

- Maximum discount
- Minimum recovery probability
- Whether discounts are allowed
- Retry limits

## Explainability

REVIVE provides an AI decision trace showing:

- Payment characteristics
- Customer behaviour
- ML prediction
- Strategies evaluated
- Merchant policies
- Selected action
- Expected revenue

## Production Integration

The prototype simulates payment execution.

In production, failed payments can enter the system through
Razorpay payment webhooks.

Razorpay Payment Links can be used for payment recovery.

Webhook events can then close the recovery loop.

## Prototype Limitations

The current buildathon prototype uses:

- Synthetic training data
- Simulated customer outcomes
- Simulated action execution
- Gradio dashboard

These components can be replaced by production APIs,
merchant data and messaging/payment systems.

## Key Innovation

REVIVE does not simply predict payment recovery.

It answers:

"Given this failed payment, what is the best action
I can take right now to maximize expected recovered
revenue while respecting merchant policy?"

## Tech Stack

Python
Pandas
NumPy
Scikit-learn
Gradio
Machine Learning
Agentic Decision Engine
Revenue Optimization
"""

with open(
    "REVIVE_README.md",
    "w",
    encoding="utf-8"
) as f:

    f.write(README)

print("✅ REVIVE_README.md created")

✅ REVIVE_README.md created
